# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

> **Revision note (engineering hardening, after the recorded Udacity run of 2026-11-12).**
> The code in this notebook was revised after the run whose outputs are preserved in `evidence/starter.executed-2026-11-12.ipynb`:
> explicit execution modes (`PIPELINE_MODE`: INITIALIZE / LOAD / REBUILD) so that a re-run never drops populated tables, update statements for
> corrected source records, deterministic tie-breaks in `dim_customer` / `dim_product`, and a stricter diagram check. Cells tagged `importable`
> are exercised by the automated tests in `tests/`. **This revised version has not been re-run on Amazon Redshift**; its saved outputs were
> therefore cleared, and the reports in `reports/` still describe the recorded run.


---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [ ]:
# ========= Imports
import os, io, re, sys, time, json, textwrap, getpass
from datetime import datetime, timezone
from decimal import Decimal
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.concurrent import execute_concurrent_with_args
from cassandra.query import dict_factory, UNSET_VALUE
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3
import botocore.exceptions

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [ ]:
# AWS credentials are read from the ENVIRONMENT only (Udacity "Cloud Resources" tab -> copy the three values).
# Nothing is hard-coded and nothing secret is printed or saved in this notebook.
# If they are missing, you are prompted with hidden input (getpass), which is not stored in the notebook output.
_AWS_VARS = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN")
_missing = [k for k in _AWS_VARS if not os.environ.get(k)]
if _missing:
    try:
        for k in _missing:
            os.environ[k] = getpass.getpass(f"{k}: ")
    except Exception:
        print("AWS credentials not set. Export", ", ".join(_missing),
              "in the environment (or re-run this cell interactively) before the Redshift tasks.")
print("AWS credential variables present:", {k: bool(os.environ.get(k)) for k in _AWS_VARS})

In [ ]:
# ========= Configuration
BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS  = os.path.join(DATA_DIR, "ecom_orders_postgres.csv")
CSV_EVENTS  = os.path.join(DATA_DIR, "ecom_events_cassandra.csv")
CSV_EDGES   = os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv")
DDL_MD_PATH = os.path.join(BASE_DIR, "project-ddl-long.md")
MERMAID_MD  = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
BATCH_SIZE  = int(os.getenv("BATCH_SIZE", "1000"))

# Execution mode of the warehouse DDL step (see "Task 4.2"):
#   INITIALIZE (default) create only the project tables that are missing; NEVER drops anything
#   LOAD                 tables must already exist; the DDL step creates nothing
#   REBUILD              drop and re-create every project table (destructive; needs the two safeguards below)
PIPELINE_MODE = os.getenv("PIPELINE_MODE", "INITIALIZE").strip().upper()
# REBUILD only runs in a database named here AND when REBUILD_CONFIRM is exactly "REBUILD <database>".
REBUILD_ALLOWED_DATABASES = {d.strip() for d in os.getenv("REBUILD_ALLOWED_DATABASES", "ecom").split(",") if d.strip()}
REBUILD_CONFIRM = os.getenv("REBUILD_CONFIRM", "")

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

# Cassandra
CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

# AWS/Redshift
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

# Verify configuration
print("Configuration loaded!")
print(f"   - BASE_DIR: {BASE_DIR}")
print(f"   - PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   - Cassandra: {CAS_HOSTS}:{CAS_PORT}/{CAS_KEYSPACE}")
print(f"   - Neo4j: {NEO4J_URI}")
print(f"   - Redshift: {REDSHIFT_DATABASE} (workgroup: {REDSHIFT_WORKGROUP})")
print(f"   - Warehouse DDL mode: {PIPELINE_MODE}")

print()
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    print(f"   AWS credentials found (Key ID: {AWS_ACCESS_KEY_ID[:4]}... (masked))")
else:
    print("   WARNING: AWS credentials NOT FOUND - set them above!")

---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.
Use these as a reference when building your transformation logic.

In [ ]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [ ]:
def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object', 'string']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS)
    events = pd.read_csv(CSV_EVENTS)
    edges  = pd.read_csv(CSV_EDGES)
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

In [ ]:
# ========= Pipeline helpers: type coercion + execution log (metadata logging)

PIPELINE_LOG: List[Dict[str, Any]] = []

class stage:
    """Context manager: records stage name, start/end, rows in/out, status and error text in PIPELINE_LOG.
    Errors are re-raised, so a failed stage can never be followed by a false success message."""
    def __init__(self, name: str, rows_in=None):
        self.rec = dict(stage=name, start=None, end=None, rows_in=rows_in, rows_out=None, status="RUNNING", error=None)
    def __enter__(self):
        self.rec["start"] = datetime.now(timezone.utc)
        PIPELINE_LOG.append(self.rec)
        return self
    def set_out(self, n):
        self.rec["rows_out"] = None if n is None else int(n)
    def __exit__(self, et, ev, tb):
        self.rec["end"] = datetime.now(timezone.utc)
        if et is None:
            self.rec["status"] = "OK"
        else:
            self.rec["status"] = "FAILED"
            self.rec["error"] = f"{et.__name__}: {ev}"
        return False

def pipeline_log_df() -> pd.DataFrame:
    df = pd.DataFrame(PIPELINE_LOG)
    if len(df):
        df["seconds"] = (df["end"] - df["start"]).dt.total_seconds().round(2)
    return df

def _py(v):
    """Convert numpy/pandas scalars (and NaN/NaT/NA) to plain Python values (None for missing)."""
    if v is None:
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return None if np.isnan(v) else float(v)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v

def df_records(df: pd.DataFrame, cols: List[str]) -> List[tuple]:
    return [tuple(_py(v) for v in row) for row in df[cols].itertuples(index=False, name=None)]

def _to_obj(s: pd.Series) -> pd.Series:
    """string/NA series -> object series where missing values are plain None."""
    return pd.Series([None if pd.isna(v) else v for v in s], index=s.index, dtype=object)

_NULL_TOKENS = ['', 'nan', 'NaN', 'None', 'null', 'NULL']
_BOOL_MAP = {'true': True, 'false': False, 't': True, 'f': False, '1': True, '0': False, 'yes': True, 'no': False}

def coerce_by_kind(df: pd.DataFrame, colspec: List[Tuple[str, str]]) -> pd.DataFrame:
    """Return a new frame with exactly the colspec columns, converted to s/ts/i/f/b kinds.
    Genuine NULLs stay NULL (never the string 'nan'); un-parseable non-null values raise instead of being dropped."""
    out = pd.DataFrame(index=df.index)
    for c, k in colspec:
        s = df[c]
        if k == 's':
            s = s.astype('string').str.strip()
            out[c] = _to_obj(s.mask(s.isin(_NULL_TOKENS)))
        elif k == 'ts':
            parsed = pd.to_datetime(s, errors='coerce')
            bad = s.notna() & parsed.isna()
            if bad.any():
                raise ValueError(f"{c}: {int(bad.sum())} un-parseable timestamp value(s), e.g. {s[bad].iloc[0]!r}")
            out[c] = parsed
        elif k in ('i', 'f'):
            num = pd.to_numeric(s, errors='coerce')
            bad = s.notna() & num.isna()
            if bad.any():
                raise ValueError(f"{c}: {int(bad.sum())} non-numeric value(s), e.g. {s[bad].iloc[0]!r}")
            out[c] = num.astype('Int64') if k == 'i' else num.astype('float64')
        elif k == 'b':
            if s.dtype == bool:
                out[c] = s.astype('boolean')
            else:
                m = s.astype('string').str.strip().str.lower().map(_BOOL_MAP)
                bad = s.notna() & m.isna()
                if bad.any():
                    raise ValueError(f"{c}: {int(bad.sum())} un-parseable boolean value(s), e.g. {s[bad].iloc[0]!r}")
                out[c] = m.astype('boolean')
        else:
            raise ValueError(f"unknown kind {k!r} for column {c}")
    return out

print("Pipeline helpers defined: stage(), pipeline_log_df(), coerce_by_kind(), df_records()")

---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [ ]:
# Read the CSV files
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()

print("\n" + "="*60)
print("TASK 1: Data Exploration")
print("="*60)

def profile(df: pd.DataFrame, name: str, key: str, ts_cols: List[str]):
    """Print/display a compact profile: shape, dtypes, nulls, duplicates, uniqueness, timestamps."""
    print(f"   Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(f"   Full-row duplicates: {int(df.duplicated().sum())}   |   unique {key}: {df[key].nunique():,} (key is unique: {df[key].is_unique})")
    prof = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "nulls": df.isna().sum(),
        "unique": df.nunique(),
        "example": df.iloc[0].astype(str),
    })
    display(prof)
    for c in ts_cols:
        t = pd.to_datetime(df[c], errors="coerce")
        print(f"   {c}: {t.min()} -> {t.max()}  (unparseable: {int(t.isna().sum() - df[c].isna().sum())})")
    print("   Sample rows:")
    display(df.head(5))

print(f"\n📊 ORDERS DATA (from PostgreSQL)")
profile(orders_df, "orders", "order_id", ["order_datetime", "ship_datetime"])

In [ ]:
print(f"\n📊 EVENTS DATA (from Cassandra)")
profile(events_df, "events", "event_id", ["event_ts"])
print("\n   event_type frequencies:")
display(events_df["event_type"].value_counts().to_frame("events"))

In [ ]:
print(f"\n📊 GRAPH EDGES DATA (from Neo4j)")
profile(edges_df, "edges", "edge_id", ["timestamp"])
print("\n   Edge shapes (from_node_type -[relationship]-> to_node_type):")
display(edges_df.groupby(["from_node_type", "relationship", "to_node_type"]).size().to_frame("edges"))

In [ ]:
print("\n" + "="*60)
print("KEY FIELDS AND RELATIONSHIPS")
print("="*60)

print("\n🔑 Primary Keys (grain of each source):")
for name, df, key in [("orders (PostgreSQL)", orders_df, "order_id"),
                      ("events (Cassandra)", events_df, "event_id"),
                      ("edges (Neo4j)", edges_df, "edge_id")]:
    print(f"   {name:22s} {key:9s} rows={len(df):,}  unique={df[key].nunique():,}  -> grain: one row per {key.replace('_id','')}")

print("\n🔗 Foreign Key Relationships / identifier overlap between sources:")
oc, ec = set(orders_df.customer_id.dropna()), set(events_df.customer_id.dropna())
cust_nodes = pd.concat([edges_df.loc[edges_df.from_node_type == "Customer", "from_node_id"],
                        edges_df.loc[edges_df.to_node_type == "Customer", "to_node_id"]])
gc = set(cust_nodes.dropna())
print(f"   Customers: orders={len(oc):,}  events={len(ec):,}  graph={len(gc):,}  union={len(oc|ec|gc):,}")
print(f"   Overlap  : orders&events={len(oc&ec)}  orders&graph={len(oc&gc)}  events&graph={len(ec&gc)}  all three={len(oc&ec&gc)}")
prod_nodes = pd.concat([edges_df.loc[edges_df.from_node_type == "Product", "from_node_id"],
                        edges_df.loc[edges_df.to_node_type == "Product", "to_node_id"], edges_df.product_id])
ep, gp = set(events_df.product_id.dropna()), set(prod_nodes.dropna())
print(f"   Products : events={len(ep):,}  graph={len(gp):,}  union={len(ep|gp):,}  overlap={len(ep&gp)}")
edge_orders = set(edges_df.order_id.dropna())
print(f"   edges.order_id -> orders.order_id: {len(edge_orders)} distinct, orphans={len(edge_orders - set(orders_df.order_id))}")
print("   Graph node routing: Customer nodes -> dim_customer, Product nodes -> dim_product (decided by *_node_type, never by ID alone)")
cp = edges_df[(edges_df.from_node_type == "Customer") & (edges_df.to_node_type == "Product")]
pp = edges_df[(edges_df.from_node_type == "Product") & (edges_df.to_node_type == "Product")]
print(f"   edges.product_id == to_node_id for Customer->Product edges: {(cp.product_id == cp.to_node_id).all()}")
print(f"   edges.product_id == from_node_id for Product->Product edges: {(pp.product_id == pp.from_node_id).all()}")

print("\n📅 Date/Time Fields (need dim_date lookups):")
for src, cols in [("orders", ["order_datetime", "ship_datetime"]), ("events", ["event_ts"]), ("edges", ["timestamp"])]:
    print(f"   {src:7s}: {cols}  -> date_key = YYYYMMDD of the date part")

print("\n📋 Categorical Fields (potential dimensions) and cardinality:")
cat_rows = []
for src, df, cols in [("orders", orders_df, ["channel", "device_type", "browser", "payment_method", "campaign", "shipping_method", "primary_category", "country", "state"]),
                      ("events", events_df, ["event_type", "device_type", "browser", "os", "referrer", "ab_variant", "category", "promo_code", "payment_outcome"]),
                      ("edges", edges_df, ["relationship", "customer_segment", "price_bucket", "region", "campaign", "category"])]:
    for c in cols:
        cat_rows.append((src, c, df[c].nunique(), int(df[c].isna().sum())))
display(pd.DataFrame(cat_rows, columns=["source", "column", "distinct_values", "nulls"]))

In [ ]:
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

key_cols = {"orders": (orders_df, ["order_id", "customer_id", "order_datetime", "order_total_usd"]),
            "events": (events_df, ["event_id", "customer_id", "session_id", "event_ts"]),
            "edges":  (edges_df,  ["edge_id", "from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship", "timestamp"])}
rows = []
for src, (df, cols) in key_cols.items():
    for c in cols:
        rows.append((src, c, int(df[c].isna().sum())))
print("Null counts in key columns (all should be 0):")
display(pd.DataFrame(rows, columns=["source", "column", "nulls"]).pivot(index="column", columns="source", values="nulls").astype("Int64"))

obs = []
obs.append(f"orders: {int(orders_df.campaign.isna().sum())} orders have NULL campaign (organic) -> kept as NULL, dim_campaign has no 'unknown' row; fact campaign_sk is NULL")
obs.append(f"events: product_id/category NULL on {int(events_df.product_id.isna().sum())} non-product events (page_view, checkout...) -> fact_events.product_sk NULL by design")
obs.append(f"events: state NULL on {int(events_df.state.isna().sum())} rows, payment_outcome NULL on {int(events_df.payment_outcome.isna().sum())} rows (only set on payment events)")
obs.append(f"edges: order_id NULL on {int(edges_df.order_id.isna().sum())} edges (VIEWED / ADDED_TO_CART / REFERRED_FRIEND / ALSO_BOUGHT_WITH carry no order)")
obs.append(f"orders: negative totals={int((orders_df.order_total_usd < 0).sum())}, ship before order={int((pd.to_datetime(orders_df.ship_datetime) < pd.to_datetime(orders_df.order_datetime)).sum())}")
obs.append(f"events: negative cart values={int((events_df.cart_value_usd < 0).sum())}; edges: non-positive unit price={int((edges_df.unit_price_usd <= 0).sum())}")
obs.append(f"cross-source customer overlap is small ({len(oc&ec)} customers appear in both orders and events) -> cross-system customer analytics cover only those customers")
obs.append("promo_code uses the literal 'NONE' for no promo: a real value, not a missing one")
print("\nObservations / data-quality issues found:")
for o in obs:
    print("  -", o)

### Task 1 deliverable: source-to-target field mapping and ID conformance

Every staging column is loaded 1:1 from the extracted/transformed source column. The table below lists, for every source field,
how it is transformed and where it lands in the **dimensional model** (the `dw` schema). Date keys are derived (`YYYYMMDD`),
surrogate keys are looked up from the dimensions.

In [ ]:
# Field mapping: (source system, source field, transformation, target table, target column)
M = []
def m(sys_, field, tf, table, col): M.append((sys_, field, tf, table, col))

PG, CA, N4 = "PostgreSQL", "Cassandra", "Neo4j"
# ---- PostgreSQL orders -> stg.orders_raw -> dw.fact_orders + dimensions
m(PG, "order_id", "trim, upper-case, validate ^ORD\\d+$, must be unique", "fact_orders", "order_id")
m(PG, "customer_id", "trim, upper-case, validate ^C\\d+$ (conformed key shared by all 3 sources)", "dim_customer", "customer_id")
m(PG, "customer_id", "surrogate-key lookup (current row)", "fact_orders", "customer_sk")
m(PG, "order_datetime", "parse timestamp; derive YYYYMMDD", "fact_orders", "order_date_key (-> dim_date.date_key)")
m(PG, "ship_datetime", "parse timestamp; derive YYYYMMDD", "fact_orders", "ship_date_key (-> dim_date.date_key)")
m(PG, "order_datetime/ship_datetime", "distinct dates", "dim_date", "date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend")
m(PG, "channel", "trim, lower-case; distinct -> surrogate key", "dim_channel / fact_orders", "channel / channel_sk")
m(PG, "device_type", "trim, lower-case; distinct -> surrogate key", "dim_device / fact_orders", "device_type / device_sk")
m(PG, "browser", "trim; distinct -> surrogate key", "dim_browser / fact_orders", "browser / browser_sk")
m(PG, "country, state", "trim, upper-case; customer geography (orders take precedence over events)", "dim_customer", "country, state")
m(PG, "payment_method", "trim, lower-case; distinct -> surrogate key", "dim_payment_method / fact_orders", "payment_method / payment_method_sk")
m(PG, "campaign", "trim; NULL stays NULL; distinct -> surrogate key", "dim_campaign / fact_orders", "campaign / campaign_sk")
m(PG, "shipping_method", "trim, lower-case; distinct -> surrogate key", "dim_shipping_method / fact_orders", "shipping_method / shipping_method_sk")
m(PG, "primary_category", "trim (degenerate attribute)", "fact_orders", "primary_category")
for f in ["num_distinct_items", "subtotal_usd", "discount_rate", "discount_amount_usd", "shipping_cost_usd", "tax_rate",
          "tax_amount_usd", "order_total_usd", "order_weight_kg", "delivery_days", "on_time_delivery", "authorization_approved", "returned"]:
    m(PG, f, "numeric/boolean conversion; rounded to DDL precision", "fact_orders", f)
# ---- Cassandra events -> stg.events_raw -> dw.fact_events
m(CA, "event_id", "trim, upper-case, validate ^EVT\\d+$, unique", "fact_events", "event_id")
m(CA, "customer_id", "trim, upper-case, validate ^C\\d+$", "dim_customer", "customer_id")
m(CA, "customer_id", "surrogate-key lookup", "fact_events", "customer_sk")
m(CA, "session_id", "trim, upper-case", "fact_events", "session_id")
m(CA, "event_type", "trim, lower-case", "fact_events", "event_type")
m(CA, "event_ts", "parse timestamp; derive YYYYMMDD", "fact_events", "event_date_key (-> dim_date.date_key)")
m(CA, "device_type", "trim, lower-case; distinct -> surrogate key", "dim_device / fact_events", "device_type / device_sk")
m(CA, "browser", "trim; distinct -> surrogate key", "dim_browser / fact_events", "browser / browser_sk")
m(CA, "os", "trim; distinct -> surrogate key", "dim_os / fact_events", "os / os_sk")
m(CA, "referrer", "trim, lower-case; distinct -> surrogate key", "dim_referrer / fact_events", "referrer / referrer_sk")
m(CA, "country, state", "fallback customer geography when no order exists", "dim_customer", "country, state")
m(CA, "ab_variant", "trim, upper-case; distinct -> surrogate key", "dim_ab_variant / fact_events", "ab_variant / ab_variant_sk")
m(CA, "is_logged_in", "boolean conversion; latest event per customer", "dim_customer", "is_logged_in")
m(CA, "product_id", "trim, upper-case, validate ^P\\d+$; surrogate-key lookup (NULL when event has no product)", "dim_product / fact_events", "product_id / product_sk")
m(CA, "category", "trim; latest per product", "dim_product / fact_events", "category / category")
for f in ["page_depth", "latency_ms", "dwell_seconds", "cart_value_usd", "discount_rate", "fraud_score", "payment_outcome", "sequence_num", "promo_code"]:
    m(CA, f, "numeric conversion / lower-case (payment_outcome) / upper-case (promo_code)", "fact_events", f)
# ---- Neo4j edges -> stg.edges_raw -> dw.fact_graph_edges
m(N4, "edge_id", "relationship property; trim, upper-case, validate ^EDGE\\d+$, unique", "fact_graph_edges", "edge_id")
m(N4, "from_node_id + from_node_type", "type=Customer -> customer_sk lookup; type=Product -> product_sk lookup", "fact_graph_edges", "from_customer_sk / from_product_sk")
m(N4, "to_node_id + to_node_type", "type=Customer -> customer_sk lookup; type=Product -> product_sk lookup", "fact_graph_edges", "to_customer_sk / to_product_sk")
m(N4, "from_node_id / to_node_id (Customer nodes)", "distinct customer ids", "dim_customer", "customer_id")
m(N4, "from_node_id / to_node_id / product_id (Product nodes)", "distinct product ids", "dim_product", "product_id")
m(N4, "relationship", "upper-case, whitelisted relationship type", "fact_graph_edges", "relationship")
m(N4, "timestamp", "parse timestamp; derive YYYYMMDD", "fact_graph_edges", "event_date_key (-> dim_date.date_key)")
m(N4, "order_id", "trim, upper-case; NULL when no order (kept as degenerate key)", "fact_graph_edges", "order_id")
m(N4, "category", "trim", "fact_graph_edges / dim_product", "category")
m(N4, "customer_segment", "trim; latest per customer (Customer source nodes)", "dim_customer / fact_graph_edges", "customer_segment")
m(N4, "price_bucket", "trim; latest per product", "dim_product / fact_graph_edges", "price_bucket")
m(N4, "unit_price_usd", "numeric conversion; latest per product -> current_unit_price_usd", "dim_product / fact_graph_edges", "current_unit_price_usd / unit_price_usd")
m(N4, "campaign", "trim; distinct -> surrogate key", "dim_campaign / fact_graph_edges", "campaign / campaign_sk")
for f in ["edge_strength", "region", "state", "prior_interactions", "dwell_seconds", "quantity", "returned_flag", "auth_approved"]:
    m(N4, f, "type conversion", "fact_graph_edges", f)
m(N4, "same_household", "boolean conversion", "stg.edges_raw only", "same_household (not carried into the fact: no DDL column)")

FIELD_MAP = pd.DataFrame(M, columns=["Source System", "Source Field", "Transformation", "Target Table (dw.)", "Target Column"])
# attach the source dtype actually observed in the CSVs
_dt = {"PostgreSQL": orders_df, "Cassandra": events_df, "Neo4j": edges_df}
FIELD_MAP.insert(2, "Source Type", [str(_dt[s][f].dtype) if f in _dt[s].columns else "-" for s, f in zip(FIELD_MAP["Source System"], FIELD_MAP["Source Field"])])

# completeness: every column of every CSV must be mentioned in the mapping
for sysname, df in _dt.items():
    mapped = " ".join(FIELD_MAP.loc[FIELD_MAP["Source System"] == sysname, "Source Field"])
    unmapped = [c for c in df.columns if not re.search(rf"\b{re.escape(c)}\b", mapped)]
    print(f"{sysname:11s}: {len(df.columns)} source columns, unmapped = {unmapped}")
    assert not unmapped, f"Unmapped {sysname} columns: {unmapped}"

print(f"\nField mapping rows: {len(FIELD_MAP)}")
pd.set_option("display.max_colwidth", 120, "display.width", 250)
display(FIELD_MAP)

print("""
ID CONFORMANCE STRATEGY
- customer_id  : same 'C#####' format in orders, events and graph Customer nodes -> trimmed + upper-cased + regex-validated, then used as the
                 natural key of dim_customer (one row per customer_id across all sources, so a customer never splits into several members).
- product_id   : same 'P####' format in events and graph Product nodes -> natural key of dim_product (union of both sources).
- graph nodes  : from_node_id/to_node_id are routed by from_node_type/to_node_type. A node id is NEVER assumed to be a customer.
- order_id     : the shared key between orders and graph edges (edges.order_id is a degenerate key on fact_graph_edges, not a surrogate lookup).
- unknown / missing ids: a NULL key is rejected during transformation (critical keys); optional foreign keys (product on a page_view,
                 campaign on organic orders) stay NULL in the fact rather than being replaced by an invented member.
- source precedence for customer attributes: country/state from orders, else events (latest record); customer_segment from graph edges;
                 is_logged_in from events (latest).""")

---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [ ]:
# Review the DDL file
print("="*60)
print("TASK 2: Warehouse Schema Design")
print("="*60)

print("\n📄 Reading DDL file:", DDL_MD_PATH)
with open(DDL_MD_PATH, "r") as f:
    ddl_content = f.read()

def ddl_statements(md: str) -> List[str]:
    """Extract ```sql fenced blocks, drop '--' comments, split into individual statements.
    Markdown prose outside the fences is never executed."""
    blocks = re.findall(r"```sql(.*?)```", md, flags=re.DOTALL | re.IGNORECASE)
    stmts = []
    for b in blocks:
        b = re.sub(r"--[^\n]*", "", b)
        stmts += [s.strip() for s in b.split(";") if s.strip()]
    return stmts

def parse_ddl(stmts: List[str]) -> pd.DataFrame:
    """One row per CREATE TABLE: schema, table, kind, columns, distribution and sort definition."""
    rows = []
    for s in stmts:
        mt = re.match(r"CREATE TABLE\s+(\w+)\.(\w+)\s*\(", s, flags=re.IGNORECASE)
        if not mt:
            continue
        schema, table = mt.groups()
        depth, end = 1, mt.end()                    # find the parenthesis that closes the column list (DECIMAL(12,2) is nested)
        while depth and end < len(s):
            depth += {"(": 1, ")": -1}.get(s[end], 0)
            end += 1
        body, tail = s[mt.end():end - 1], s[end:]
        cols = []
        for line in re.split(r",\s*\n", body):
            tok = line.strip().split()
            if tok and tok[0].upper() not in ("PRIMARY", "FOREIGN", "UNIQUE", "CONSTRAINT"):
                cols.append(tok[0].strip('"'))
        dist = re.search(r"DISTKEY\s*\(\s*(\w+)\s*\)", tail, re.I)
        sort = re.search(r"SORTKEY\s*\(\s*(\w+)\s*\)", tail, re.I)
        style = "ALL" if re.search(r"DISTSTYLE\s+ALL", tail, re.I) else ("KEY" if dist else "AUTO/EVEN")
        kind = "staging" if schema == "stg" else ("fact" if table.startswith("fact_") else "dimension")
        rows.append(dict(schema=schema, table=table, kind=kind, n_cols=len(cols), columns=cols, diststyle=style,
                         distkey=dist.group(1) if dist else None, sortkey=sort.group(1) if sort else None,
                         has_identity="IDENTITY" in body.upper()))
    return pd.DataFrame(rows)

DDL_STATEMENTS = ddl_statements(ddl_content)
DDL_TABLES = parse_ddl(DDL_STATEMENTS)
print(f"\nSQL statements found in fenced blocks: {len(DDL_STATEMENTS)}")
print("\nTables defined:")
display(DDL_TABLES.drop(columns="columns"))
cnt = DDL_TABLES.kind.value_counts().to_dict()
print(f"\nStaging tables: {cnt.get('staging', 0)}   Dimension tables: {cnt.get('dimension', 0)}   Fact tables: {cnt.get('fact', 0)}")
assert (cnt.get("staging"), cnt.get("dimension"), cnt.get("fact")) == (3, 12, 3), "unexpected table counts in DDL"

# Naming strategy check: the DDL uses the stg / dw schemas; the notebook uses the SAME names everywhere.
print("\nNaming strategy: schemas 'stg' and 'dw' (no public.stg_/public.dw_ prefixes). See rs_exec() for the documented fallback.")

# Join-colocation audit: a fact->dimension join is colocated only if both sides are DISTKEY on the join column.
dk = DDL_TABLES.set_index("table")
audit = []
for fact, col, dim, dimcol in [("fact_orders", "customer_sk", "dim_customer", "customer_sk"),
                               ("fact_events", "customer_sk", "dim_customer", "customer_sk"),
                               ("fact_graph_edges", "to_product_sk", "dim_product", "product_sk")]:
    ok = dk.loc[fact, "distkey"] == col and dk.loc[dim, "distkey"] == dimcol
    audit.append((f"{fact}.{col} = {dim}.{dimcol}", dk.loc[fact, "distkey"], dk.loc[dim, "distkey"], "colocated" if ok else "NOT colocated"))
print("\nJoin colocation audit (fact DISTKEY vs dimension DISTKEY):")
display(pd.DataFrame(audit, columns=["join", "fact distkey", "dim distkey", "result"]))
assert all(a[3] == "colocated" for a in audit)
print("\n✅ DDL parsed: 3 staging + 12 dimension + 3 fact tables, DISTKEYs align with the join columns")

In [ ]:
print("\n" + "="*60)
print("SCHEMA DESIGN DOCUMENTATION")
print("="*60)

# Design decisions per table (also reused in Task 5 and the final report)
DESIGN_TABLE = pd.DataFrame([
    ("dw.fact_orders",       "KEY(customer_sk)",  "order_date_key", "Customer-centric revenue queries; colocates with dim_customer (also DISTKEY customer_sk). Date sort key serves date-range filters/daily revenue."),
    ("dw.fact_events",       "KEY(customer_sk)",  "event_date_key", "Clickstream is analysed per customer/session; colocated with dim_customer. Date sort key for time windows."),
    ("dw.fact_graph_edges",  "KEY(to_product_sk)","event_date_key", "Edges are mostly Customer->Product; distributing on the target product colocates with dim_product (DISTKEY product_sk) for product analysis."),
    ("dw.dim_customer",      "KEY(customer_sk)",  "customer_sk",    "Large dimension; same key as the customer-distributed facts so fact.customer_sk = dim.customer_sk needs no redistribution."),
    ("dw.dim_product",       "KEY(product_sk)",   "product_sk",     "Large dimension; same key as fact_graph_edges.to_product_sk."),
    ("dw.dim_date",          "ALL",               "date_key",       "Tiny and joined by every fact: replicate to every node; sort by date_key."),
    ("dw.dim_channel/device/browser/os/referrer/campaign/payment_method/shipping_method/ab_variant",
                              "ALL",               "(none)",         "Tiny lookup tables (<=12 rows): DISTSTYLE ALL avoids any shuffle in star joins."),
    ("stg.orders_raw / events_raw / edges_raw", "AUTO/EVEN", "(none)", "Transient landing tables, fully re-loaded each run; no join workload, so no key tuning."),
], columns=["table", "distribution", "sort key", "justification"])

schema_doc = """
## Staging Tables (stg schema)
- stg.orders_raw  : raw PostgreSQL orders (26 columns, 1 row per order)           -> feeds dim_* and fact_orders
- stg.events_raw  : raw Cassandra clickstream events (24 columns, 1 row per event) -> feeds dim_* and fact_events
- stg.edges_raw   : raw Neo4j relationships (23 columns, 1 row per edge)           -> feeds dim_* and fact_graph_edges
  Why staging? It decouples extraction from modelling: data lands untransformed (permissive types), can be reconciled
  against the sources by row count and checksum, and the warehouse load becomes a repeatable set-based SQL step.

## Dimension Tables (dw schema)  - surrogate keys are IDENTITY columns
- dim_date (date_key = YYYYMMDD) ........ calendar attributes for order, ship, event and edge dates
- dim_customer ............................ one row per conformed customer_id across all sources; SCD-2-ready columns
                                              (effective_from/effective_to/is_current) but loaded as ONE current row per customer
- dim_product ............................. one row per product_id (events + graph Product nodes); SCD-2-ready, one current row each
- dim_channel, dim_device, dim_browser, dim_os, dim_referrer, dim_campaign, dim_payment_method,
  dim_shipping_method, dim_ab_variant ..... small reference dimensions (DISTSTYLE ALL)

## Fact Tables (dw schema)
- fact_orders      grain: ONE ROW PER ORDER (order_id unique)        measures: subtotal, discount, shipping, tax, total, weight, delivery_days, flags
- fact_events      grain: ONE ROW PER CLICKSTREAM EVENT (event_id)   measures: page_depth, latency_ms, dwell_seconds, cart_value, fraud_score ...
- fact_graph_edges grain: ONE ROW PER GRAPH EDGE (edge_id unique)    measures: edge_strength, unit_price_usd, quantity, prior_interactions ...
  Each fact row is built from exactly one staging row; dimension lookups are LEFT JOINs on unique natural keys, so rows
  can neither be multiplied nor silently dropped.

## Optimization Choices
- DISTKEY on the surrogate key of both sides of the big joins (colocated); DISTSTYLE ALL for small dimensions
- SORTKEY on the date key of every fact (date-range scans/zone maps) and on the key of the big dimensions
- ENCODE zstd on most columns (good ratio for text/numerics), ENCODE raw on the leading sort-key column
- Redshift PRIMARY KEY/FOREIGN KEY are informational only -> integrity is checked by explicit queries in Task 6

## Analytics supported
- Revenue: daily/monthly revenue, AOV, discounts, returns by channel / campaign / category / payment method   (fact_orders x dims)
- Customer behaviour: funnel and sessions by referrer / device / A-B variant, activity per customer             (fact_events x dims)
- Product performance: purchases, views, returns, co-purchase ("ALSO_BOUGHT_WITH") per product / category        (fact_graph_edges x dim_product)
"""
print(schema_doc)
pd.set_option("display.max_colwidth", 200)
display(DESIGN_TABLE)

CORRECTIONS = [
    ("D1", "dim_customer/dim_product DISTKEY moved from business key to surrogate key", "facts join on *_sk, so the original design was not colocated"),
    ("D2", "DROP ... CASCADE on dim_customer/dim_product", "lookup views depend on them; DDL is now re-runnable"),
    ("D3", "channel_sk removed from fact_events", "events have no channel field (column would always be NULL)"),
    ("D4", "stg.edges_raw.\"timestamp\" quoted", "type-name collision"),
    ("D8", "ENCODE raw on leading sort-key columns", "Redshift guidance"),
]
print("\nCorrections applied to the supplied DDL (audit):")
display(pd.DataFrame(CORRECTIONS, columns=["id", "change", "reason"]))

In [ ]:
# ========= Schema-diagram check (importable function, tested in tests/) =========
_TYPE_FAMILY = {"INT": "int", "INTEGER": "int", "SMALLINT": "int", "BIGINT": "int", "VARCHAR": "text", "DECIMAL": "num", "NUMERIC": "num",
                "DATE": "date", "TIMESTAMP": "ts", "BOOLEAN": "bool"}

def ddl_column_types(stmt: str) -> Dict[str, str]:
    """column -> type family for one CREATE TABLE statement (int / text / num / date / ts / bool)."""
    mt = re.match(r"CREATE TABLE\s+(\w+)\.(\w+)\s*\(", stmt, flags=re.I)
    depth, end = 1, mt.end()
    while depth and end < len(stmt):
        depth += {"(": 1, ")": -1}.get(stmt[end], 0)
        end += 1
    out = {}
    for line in re.split(r",\s*\n", stmt[mt.end():end - 1]):
        tok = line.strip().split()
        if len(tok) >= 2 and tok[0].upper() not in ("PRIMARY", "FOREIGN", "UNIQUE", "CONSTRAINT"):
            out[tok[0].strip('"')] = _TYPE_FAMILY.get(re.match(r"[A-Za-z]+", tok[1]).group(0).upper(), "?")
    return out

def check_diagram(ddl_stmts: List[str], ddl_tables: pd.DataFrame, mermaid_src: str) -> List[str]:
    """Compare the Mermaid ER diagram with the DDL. Returns a list of problems (empty = consistent).
    Checks: dw tables missing/extra; columns missing/extra per table; type FAMILY per column (int/text/num/date/ts/bool, not precision);
    every *_sk / *_date_key column of a fact has a relationship line; every relationship names a real fact column whose name ends with the
    dimension's key column. NOT checked: DISTKEY/SORTKEY/ENCODE (verified against the Redshift catalog in Task 4), precision/length, NOT NULL."""
    problems = []
    types = {}
    for s in ddl_stmts:
        mt = re.match(r"CREATE TABLE\s+(\w+)\.(\w+)\s*\(", s, flags=re.I)
        if mt and mt.group(1) == "dw":
            types[mt.group(2)] = ddl_column_types(s)
    mm_tables = dict(re.findall(r"^\s*dw_(\w+)\s*\{(.*?)\}", mermaid_src, flags=re.S | re.M))
    for table, cols in types.items():
        if table not in mm_tables:
            problems.append(f"table missing in diagram: dw_{table}")
            continue
        mm_cols = {}
        for typ, name in re.findall(r"\b(INT|BIGINT|VARCHAR|DECIMAL|DATE|SMALLINT|BOOLEAN|TIMESTAMP)\s+(\w+)", mm_tables[table]):
            mm_cols[name] = _TYPE_FAMILY[typ]
        for c in sorted(set(cols) - set(mm_cols)):
            problems.append(f"dw_{table}: column {c} is in the DDL but not in the diagram")
        for c in sorted(set(mm_cols) - set(cols)):
            problems.append(f"dw_{table}: column {c} is in the diagram but not in the DDL")
        for c in sorted(set(cols) & set(mm_cols)):
            if cols[c] != mm_cols[c]:
                problems.append(f"dw_{table}.{c}: DDL type family {cols[c]!r} vs diagram {mm_cols[c]!r}")
    for table in sorted(set(mm_tables) - set(types)):
        problems.append(f"diagram has table not in DDL: dw_{table}")
    rels = re.findall(r"^\s*dw_(\w+)\s+\|\|--o\{\s+dw_(\w+)\s*:\s*\"(\w+)\"", mermaid_src, flags=re.M)
    dim_key = {t: next(iter(cols)) for t, cols in types.items() if t.startswith("dim_")}
    for dim, fact, col in rels:
        if dim not in dim_key or fact not in types:
            problems.append(f"relationship {dim} -> {fact} names an unknown table")
        elif col not in types[fact]:
            problems.append(f"relationship {dim} -> {fact}: {col!r} is not a column of {fact}")
        elif not col.endswith(dim_key[dim]):
            problems.append(f"relationship {dim} -> {fact}: column {col!r} does not end with the dimension key {dim_key[dim]!r}")
    have = {(fact, col) for _, fact, col in rels}
    for table, cols in types.items():
        if table.startswith("fact_"):
            own_key = next(iter(cols))                     # the fact's own surrogate key (order_sk, event_sk, edge_sk)
            for c in cols:
                if c != own_key and (c.endswith("_sk") or c.endswith("_date_key")) and (table, c) not in have:
                    problems.append(f"{table}.{c}: key column without a relationship line in the diagram")
    return problems


In [ ]:
# Schema diagram: the Mermaid ER diagram in project-mermaid-diagram.md must match the DDL. Verify it programmatically.
with open(MERMAID_MD) as f:
    mermaid_src = f.read()
problems = check_diagram(DDL_STATEMENTS, DDL_TABLES, mermaid_src)
print("Diagram vs DDL check:", "MATCH (tables, columns, type families and fact-to-dimension relationships; see check_diagram() for what is NOT checked)" if not problems else problems)
assert not problems
display(Markdown("**Schema diagram** (rendered image: `reports/schema-diagram.svg`; Mermaid source: `project-mermaid-diagram.md`). "
                 "Logical names are `dw.<table>`; the diagram writes them `dw_<table>`. In the Udacity lab the same tables were physically named `public.dw_<table>`.\n\n"
                 "![schema](reports/schema-diagram.svg)"))


---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [ ]:
# ========= PostgreSQL Functions =========
PG_TYPES = {'s': 'VARCHAR(64)', 'ts': 'TIMESTAMP', 'i': 'INTEGER', 'f': 'DOUBLE PRECISION', 'b': 'BOOLEAN'}

def pg_connect():
    """Connect to PostgreSQL."""
    try:
        return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PW, connect_timeout=10)
    except psycopg2.OperationalError as e:
        raise ConnectionError(f"PostgreSQL connection failed ({PG_HOST}:{PG_PORT}/{PG_DB} as {PG_USER}): {e}") from e

def pg_load_orders(df: pd.DataFrame) -> int:
    """Create raw.orders and upsert the orders. Re-runnable: new orders are inserted, an order whose values changed is updated,
    an unchanged order is left alone (no write). Orders missing from the CSV are NOT deleted. Returns the row count afterwards."""
    typed = coerce_by_kind(df, ORDERS_COLSPEC)
    cols = [c for c, _ in ORDERS_COLSPEC]
    rows = df_records(typed, cols)
    col_ddl = ",\n    ".join(f"{c} {PG_TYPES[k]}" + (" PRIMARY KEY" if c == "order_id" else "") for c, k in ORDERS_COLSPEC)
    conn = pg_connect()
    try:
        with conn:                                   # commit on success, rollback on error
            with conn.cursor() as cur:
                cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
                cur.execute(f"CREATE TABLE IF NOT EXISTS raw.orders (\n    {col_ddl}\n);")
                value_cols = [c for c in cols if c != "order_id"]
                sets = ", ".join(f"{c} = EXCLUDED.{c}" for c in value_cols)
                changed = " OR ".join(f"t.{c} IS DISTINCT FROM EXCLUDED.{c}" for c in value_cols)
                execute_values(cur, f"INSERT INTO raw.orders AS t ({', '.join(cols)}) VALUES %s "
                                    f"ON CONFLICT (order_id) DO UPDATE SET {sets} WHERE {changed}",
                               rows, page_size=BATCH_SIZE)
                cur.execute("SELECT COUNT(*) FROM raw.orders")
                return cur.fetchone()[0]
    finally:
        conn.close()

def extract_from_pg() -> pd.DataFrame:
    """Extract orders from PostgreSQL (through SQLAlchemy + pandas)."""
    url = URL.create("postgresql+psycopg2", username=PG_USER, password=PG_PW, host=PG_HOST, port=PG_PORT, database=PG_DB)
    engine = create_engine(url)
    try:
        cols = ", ".join(c for c, _ in ORDERS_COLSPEC)
        return pd.read_sql_query(f"SELECT {cols} FROM raw.orders ORDER BY order_id", engine)
    finally:
        engine.dispose()

print("PostgreSQL functions defined: pg_connect(), pg_load_orders(), extract_from_pg()")

In [ ]:
# ========= Cassandra Functions =========
CAS_TYPES = {'s': 'text', 'ts': 'timestamp', 'i': 'int', 'f': 'double', 'b': 'boolean'}

def cas_connect():
    """Connect to Cassandra and ensure the keyspace exists. Returns (session, cluster)."""
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CAS_KEYSPACE):
        raise ValueError(f"unsafe keyspace name: {CAS_KEYSPACE!r}")
    auth = PlainTextAuthProvider(CAS_USER, CAS_PW) if CAS_USER else None
    cluster = Cluster(CAS_HOSTS, port=CAS_PORT, auth_provider=auth, connect_timeout=15)
    try:
        session = cluster.connect()
    except Exception as e:
        cluster.shutdown()
        raise ConnectionError(f"Cassandra connection failed ({CAS_HOSTS}:{CAS_PORT}): {e}") from e
    session.execute(f"CREATE KEYSPACE IF NOT EXISTS {CAS_KEYSPACE} "
                    "WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}")
    session.set_keyspace(CAS_KEYSPACE)
    return session, cluster

def cas_load_events(df: pd.DataFrame) -> int:
    """Create the events table and load the events with prepared statements + bounded concurrency.
    Primary key ((customer_id), event_ts, event_id): partition by customer (typical clickstream access pattern),
    cluster by time, event_id keeps rows unique. INSERTs are upserts, so re-runs do not duplicate rows."""
    typed = coerce_by_kind(df, EVENTS_COLSPEC)
    for k in ("event_id", "customer_id", "event_ts"):
        if typed[k].isna().any():
            raise ValueError(f"cannot load events: primary-key column {k} contains NULLs")
    cols = [c for c, _ in EVENTS_COLSPEC]
    session, cluster = cas_connect()
    try:
        col_ddl = ",\n    ".join(f"{c} {CAS_TYPES[k]}" for c, k in EVENTS_COLSPEC)
        session.execute(f"CREATE TABLE IF NOT EXISTS events (\n    {col_ddl},\n"
                        "    PRIMARY KEY ((customer_id), event_ts, event_id)\n) WITH CLUSTERING ORDER BY (event_ts DESC, event_id ASC)")
        prep = session.prepare(f"INSERT INTO events ({', '.join(cols)}) VALUES ({', '.join(['?'] * len(cols))})")
        # NULL -> UNSET_VALUE: do not write tombstones for missing values
        params = [tuple(UNSET_VALUE if v is None else v for v in row) for row in df_records(typed, cols)]
        results = execute_concurrent_with_args(session, prep, params, concurrency=32, raise_on_first_error=False)
        failed = [r for ok, r in results if not ok]
        if failed:
            raise RuntimeError(f"{len(failed)} of {len(params)} Cassandra inserts failed; first error: {failed[0]!r}")
        return session.execute("SELECT COUNT(*) FROM events").one()[0]
    finally:
        cluster.shutdown()

def extract_from_cas() -> pd.DataFrame:
    """Extract all events from Cassandra as a DataFrame."""
    session, cluster = cas_connect()
    try:
        session.row_factory = dict_factory
        session.default_fetch_size = 1000
        cols = [c for c, _ in EVENTS_COLSPEC]
        rows = list(session.execute(f"SELECT {', '.join(cols)} FROM events"))
        return pd.DataFrame(rows, columns=cols).sort_values("event_id").reset_index(drop=True)
    finally:
        cluster.shutdown()

print("Cassandra functions defined: cas_connect(), cas_load_events(), extract_from_cas()")

In [ ]:
# ========= Neo4j Functions =========
# Labels and relationship types cannot be Cypher parameters, so they are checked against fixed whitelists
# before being placed in the query text. Nothing else from the data is ever concatenated into Cypher.
ALLOWED_NODE_LABELS = {"Customer", "Product", "Order"}
ALLOWED_REL_TYPES = {"VIEWED", "PURCHASED", "ADDED_TO_CART", "ALSO_BOUGHT_WITH", "REFERRED_FRIEND", "RETURNS"}
_EDGE_STRUCT_COLS = ("from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship")
_EDGE_PROP_SPEC = [(c, k) for c, k in EDGES_COLSPEC if c not in _EDGE_STRUCT_COLS]   # stored as relationship properties

def neo4j_driver():
    """Connect to Neo4j and verify connectivity."""
    drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))
    try:
        drv.verify_connectivity()
    except Exception as e:
        drv.close()
        raise ConnectionError(f"Neo4j connection failed ({NEO4J_URI}): {e}") from e
    return drv

def _edge_props(row: dict) -> dict:
    props = {}
    for c, k in _EDGE_PROP_SPEC:
        v = _py(row[c])
        if v is None:
            continue                                  # missing values are simply not stored on the relationship
        props[c] = v.strftime("%Y-%m-%d %H:%M:%S") if k == "ts" else v   # timestamps kept as exact source text
    return props

def neo4j_load_edges(df: pd.DataFrame, batch_size: int = 500) -> int:
    """Load edges as (node)-[relationship {edge_id, ...properties}]->(node) using MERGE (idempotent).
    Returns the number of relationships in the graph afterwards."""
    typed = coerce_by_kind(df, EDGES_COLSPEC)
    # 1. clean + validate
    bad_label = ~(typed.from_node_type.isin(ALLOWED_NODE_LABELS) & typed.to_node_type.isin(ALLOWED_NODE_LABELS))
    bad_rel = ~typed.relationship.isin(ALLOWED_REL_TYPES)
    if bad_label.any() or bad_rel.any():
        raise ValueError(f"unsupported node types {sorted(set(typed.loc[bad_label, 'from_node_type']) | set(typed.loc[bad_label, 'to_node_type']))} "
                         f"or relationship types {sorted(set(typed.loc[bad_rel, 'relationship']))}")
    if typed.edge_id.isna().any() or typed.edge_id.duplicated().any():
        raise ValueError("edge_id must be present and unique")
    if typed[["from_node_id", "to_node_id"]].isna().any().any():
        raise ValueError("from_node_id/to_node_id must not be NULL")
    drv = neo4j_driver()
    try:
        with drv.session() as s:
            # 2. node constraints (also create the indexes MERGE needs)
            for label in sorted(ALLOWED_NODE_LABELS):
                try:        # Neo4j 4.4+/5 syntax
                    s.run(f"CREATE CONSTRAINT {label.lower()}_id_unique IF NOT EXISTS FOR (n:{label}) REQUIRE n.id IS UNIQUE").consume()
                except Exception:   # older Neo4j 4.x syntax
                    s.run(f"CREATE CONSTRAINT IF NOT EXISTS ON (n:{label}) ASSERT n.id IS UNIQUE").consume()
            # 3. MERGE nodes and relationships, one query per (from label, relationship, to label)
            for (fl, rel, tl), g in typed.groupby(["from_node_type", "relationship", "to_node_type"]):
                cypher = (f"UNWIND $rows AS r "
                          f"MERGE (a:{fl} {{id: r.from_id}}) MERGE (b:{tl} {{id: r.to_id}}) "
                          f"MERGE (a)-[e:{rel} {{edge_id: r.edge_id}}]->(b) SET e += r.props")
                rows = [dict(from_id=r["from_node_id"], to_id=r["to_node_id"], edge_id=r["edge_id"], props=_edge_props(r))
                        for r in g.to_dict("records")]
                # 4. process in batches
                for i in range(0, len(rows), batch_size):
                    chunk = rows[i:i + batch_size]
                    s.execute_write(lambda tx, chunk=chunk: tx.run(cypher, rows=chunk).consume())
            return s.run("MATCH ()-[e]->() WHERE e.edge_id IS NOT NULL RETURN count(e) AS n").single()["n"]
    finally:
        drv.close()

def extract_from_neo4j() -> pd.DataFrame:
    """Extract relationships from Neo4j as a tabular edge list (one row per relationship)."""
    props = ", ".join(f"e.{c} AS `{c}`" for c, _ in _EDGE_PROP_SPEC)
    cypher = ("MATCH (a)-[e]->(b) WHERE e.edge_id IS NOT NULL "
              "RETURN a.id AS from_node_id, labels(a)[0] AS from_node_type, b.id AS to_node_id, "
              f"labels(b)[0] AS to_node_type, type(e) AS relationship, {props} ORDER BY e.edge_id")
    drv = neo4j_driver()
    try:
        with drv.session() as s:
            rows = [r.data() for r in s.run(cypher)]
    finally:
        drv.close()
    cols = [c for c, _ in EDGES_COLSPEC]
    return pd.DataFrame(rows).reindex(columns=cols)

print("Neo4j functions defined: neo4j_driver(), neo4j_load_edges(), extract_from_neo4j()")

## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [ ]:
print("="*60)
print("TASK 3: Loading Source Systems")
print("="*60)

source_counts = {}

print("\n📦 Loading orders into PostgreSQL...")
with stage("load PostgreSQL raw.orders", rows_in=len(orders_df)) as st:
    source_counts["PostgreSQL"] = pg_load_orders(orders_df)
    st.set_out(source_counts["PostgreSQL"])
print(f"   raw.orders now holds {source_counts['PostgreSQL']:,} rows (CSV: {len(orders_df):,})")

print("\n📦 Loading events into Cassandra...")
with stage("load Cassandra events", rows_in=len(events_df)) as st:
    source_counts["Cassandra"] = cas_load_events(events_df)
    st.set_out(source_counts["Cassandra"])
print(f"   {CAS_KEYSPACE}.events now holds {source_counts['Cassandra']:,} rows (CSV: {len(events_df):,})")

print("\n📦 Loading edges into Neo4j...")
with stage("load Neo4j edges", rows_in=len(edges_df)) as st:
    source_counts["Neo4j"] = neo4j_load_edges(edges_df)
    st.set_out(source_counts["Neo4j"])
print(f"   graph now holds {source_counts['Neo4j']:,} relationships (CSV: {len(edges_df):,})")

# Verification against the databases themselves (not against what the load functions returned)
_conn = pg_connect()
with _conn.cursor() as cur:
    cur.execute("SELECT COUNT(*), COUNT(DISTINCT order_id) FROM raw.orders"); pg_n, pg_u = cur.fetchone()
_conn.close()
_s, _c = cas_connect()
cas_n = _s.execute("SELECT COUNT(*) FROM events").one()[0]
_c.shutdown()
_d = neo4j_driver()
with _d.session() as _ns:
    n4_rel = _ns.run("MATCH ()-[e]->() RETURN count(e) AS n").single()["n"]
    n4_nodes = _ns.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n ORDER BY label").data()
    n4_types = _ns.run("MATCH ()-[e]->() RETURN type(e) AS rel, count(e) AS n ORDER BY rel").data()
_d.close()
print("\nVerification queries against the source databases:")
display(pd.DataFrame([("PostgreSQL raw.orders", pg_n, pg_u, len(orders_df)),
                      ("Cassandra events", cas_n, None, len(events_df)),
                      ("Neo4j relationships", n4_rel, None, len(edges_df))],
                     columns=["source", "rows in database", "distinct keys", "rows in CSV"]))
print("Neo4j nodes:", n4_nodes)
print("Neo4j relationship types:", {r['rel']: r['n'] for r in n4_types})
assert (pg_n, cas_n, n4_rel) == (len(orders_df), len(events_df), len(edges_df)), "source load count mismatch"
print("\n✅ All source systems loaded and verified (counts match the CSV files)")

## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [ ]:
# ========= Transformation: conform extracted data to the staging specification =========
CASE_RULES = {
    "lower": {"channel", "device_type", "payment_method", "shipping_method", "referrer", "event_type", "payment_outcome"},
    "upper": {"order_id", "customer_id", "event_id", "session_id", "product_id", "edge_id", "from_node_id", "to_node_id",
              "country", "state", "ab_variant", "promo_code", "relationship", "region"},
    "title": {"from_node_type", "to_node_type"},
}
ID_PATTERNS = {
    "order_id": r"ORD\d+", "customer_id": r"C\d+", "event_id": r"EVT\d+", "session_id": r"S\d+", "product_id": r"P\d+",
    "edge_id": r"EDGE\d+", "from_node_id": r"(?:C|P|ORD)\d+", "to_node_id": r"(?:C|P|ORD)\d+",
}
# decimal places of the staging DDL (DECIMAL(12,2) money, (5,3) rates, (6,4) tax_rate, (6,3) scores ...)
ROUND_TO = {"subtotal_usd": 2, "discount_amount_usd": 2, "shipping_cost_usd": 2, "tax_amount_usd": 2, "order_total_usd": 2,
            "order_weight_kg": 2, "cart_value_usd": 2, "unit_price_usd": 2, "discount_rate": 3, "tax_rate": 4,
            "fraud_score": 3, "edge_strength": 3}
CRITICAL_KEYS = {"orders": ["order_id", "customer_id", "order_datetime"],
                 "events": ["event_id", "customer_id", "event_ts"],
                 "edges":  ["edge_id", "from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship", "timestamp"]}
SCHEMA_DRIFT_LOG: List[str] = []

def check_schema_drift(df: pd.DataFrame, colspec, name: str) -> pd.DataFrame:
    """Practical schema-drift handling: critical columns missing -> error; optional missing -> added as NULL (logged);
    unexpected extra columns -> dropped (logged)."""
    expected = [c for c, _ in colspec]
    missing = [c for c in expected if c not in df.columns]
    extra = [c for c in df.columns if c not in expected]
    crit_missing = [c for c in missing if c in CRITICAL_KEYS[name]]
    if crit_missing:
        raise ValueError(f"[{name}] schema drift: critical column(s) missing: {crit_missing}")
    for c in missing:
        df = df.assign(**{c: np.nan})
        SCHEMA_DRIFT_LOG.append(f"[{name}] optional column '{c}' missing -> added as NULL")
    for c in extra:
        SCHEMA_DRIFT_LOG.append(f"[{name}] unexpected column '{c}' -> ignored (not in staging spec)")
    return df

def conform(df: pd.DataFrame, colspec, name: str) -> pd.DataFrame:
    """Return a DataFrame whose columns, order and dtypes match the staging table specification."""
    df = check_schema_drift(df, colspec, name)
    out = coerce_by_kind(df, colspec)
    for c, k in colspec:
        if k == "s":
            for rule, fn in (("lower", "lower"), ("upper", "upper"), ("title", "title")):
                if c in CASE_RULES[rule]:
                    out[c] = _to_obj(getattr(out[c].astype("string").str, fn)())
        elif k == "f" and c in ROUND_TO:
            out[c] = out[c].round(ROUND_TO[c])
    # validate identifiers and critical keys
    for c, pat in ID_PATTERNS.items():
        if c in out.columns:
            bad = out[c].notna() & ~out[c].astype("string").str.fullmatch(pat).fillna(False)
            if bad.any():
                raise ValueError(f"[{name}] {int(bad.sum())} malformed {c} value(s), e.g. {out.loc[bad, c].iloc[0]!r}")
    for c in CRITICAL_KEYS[name]:
        if out[c].isna().any():
            raise ValueError(f"[{name}] critical column {c} has {int(out[c].isna().sum())} NULL(s) - refusing to invent a value")
    pk = colspec[0][0]
    if out[pk].duplicated().any():
        raise ValueError(f"[{name}] duplicate {pk} values: {int(out[pk].duplicated().sum())}")
    return out

def compare_frames(a: pd.DataFrame, b: pd.DataFrame, key: str) -> pd.DataFrame:
    """Column-by-column comparison of two conformed frames (NULL-safe). Returns mismatch counts per column."""
    a = a.sort_values(key).reset_index(drop=True)
    b = b.sort_values(key).reset_index(drop=True)
    if len(a) != len(b) or not a[key].equals(b[key]):
        raise ValueError("key sets differ between the compared frames")
    rows = []
    for c in a.columns:
        x, y = a[c], b[c]
        if pd.api.types.is_float_dtype(x) or str(x.dtype) == "Float64":
            same = (x.isna() & y.isna()) | ((x - y).abs() < 1e-9)
        else:
            same = (x.isna() & y.isna()) | (x.astype(object) == y.astype(object))
        rows.append((c, int((~same.fillna(False)).sum())))
    return pd.DataFrame(rows, columns=["column", "mismatches"])

print("Transformation functions defined: check_schema_drift(), conform(), compare_frames()")

In [ ]:
print("\n" + "="*60)
print("Extracting from Source Systems")
print("="*60)

print("\n📤 Extracting from PostgreSQL...")
with stage("extract PostgreSQL") as st:
    orders_extracted = extract_from_pg(); st.set_out(len(orders_extracted))
print(f"   Extracted {len(orders_extracted):,} orders")

print("\n📤 Extracting from Cassandra...")
with stage("extract Cassandra") as st:
    events_extracted = extract_from_cas(); st.set_out(len(events_extracted))
print(f"   Extracted {len(events_extracted):,} events")

print("\n📤 Extracting from Neo4j...")
with stage("extract Neo4j") as st:
    edges_extracted = extract_from_neo4j(); st.set_out(len(edges_extracted))
print(f"   Extracted {len(edges_extracted):,} edges")

print("\n🔄 Conforming data to staging specifications...")
with stage("transform orders", rows_in=len(orders_extracted)) as st:
    orders = conform(orders_extracted, ORDERS_COLSPEC, "orders"); st.set_out(len(orders))
with stage("transform events", rows_in=len(events_extracted)) as st:
    events = conform(events_extracted, EVENTS_COLSPEC, "events"); st.set_out(len(events))
with stage("transform edges", rows_in=len(edges_extracted)) as st:
    edges = conform(edges_extracted, EDGES_COLSPEC, "edges"); st.set_out(len(edges))
print("   Schema-drift log:", SCHEMA_DRIFT_LOG or "no drift detected (all expected columns present, none unexpected)")

# The transformed frames must match the staging column specification exactly
for nm, frame, spec in [("orders", orders, ORDERS_COLSPEC), ("events", events, EVENTS_COLSPEC), ("edges", edges, EDGES_COLSPEC)]:
    assert list(frame.columns) == [c for c, _ in spec], f"{nm}: column names/order differ from COLSPEC"
print("   Column names and order match ORDERS/EVENTS/EDGES_COLSPEC ✔")

# Data extracted from the databases must equal the original CSV content (after the same conforming)
print("\n🔍 Reconciling extracted data with the original CSV files (value-by-value):")
chk = []
for nm, csv_df, got, spec, key in [("orders", orders_df, orders, ORDERS_COLSPEC, "order_id"),
                                   ("events", events_df, events, EVENTS_COLSPEC, "event_id"),
                                   ("edges", edges_df, edges, EDGES_COLSPEC, "edge_id")]:
    mism = compare_frames(conform(csv_df, spec, nm), got, key)
    chk.append((nm, len(csv_df), len(got), int(mism.mismatches.sum())))
display(pd.DataFrame(chk, columns=["dataset", "csv rows", "extracted+transformed rows", "cell mismatches"]))
assert all(r[3] == 0 and r[1] == r[2] for r in chk), "extracted data differs from the CSV source"
print("\n✅ Task 3 verified: data extracted through psycopg2/SQLAlchemy, cassandra-driver and the neo4j driver; frames conform to the staging specs")

In [ ]:
# Transformation evidence: what changed (before -> after), conformed IDs, derived fields
print("TRANSFORMATION EVIDENCE")
print("="*60)
before = {"orders": orders_df, "events": events_df, "edges": edges_df}
after = {"orders": orders, "events": events, "edges": edges}
dt_rows = []
for nm in before:
    for c in after[nm].columns:
        b, a = str(before[nm][c].dtype), str(after[nm][c].dtype)
        if b != a:
            dt_rows.append((nm, c, b, a))
print("\n1) Column type conversions (CSV string/float -> staging types):")
display(pd.DataFrame(dt_rows, columns=["dataset", "column", "dtype before", "dtype after"]))

print("2) Values changed by standardisation (case/whitespace/rounding), CSV vs transformed:")
chg = []
for nm in before:
    for c in after[nm].columns:
        b, a = before[nm][c], after[nm][c]
        if str(b.dtype) == str(a.dtype) or (b.dtype == object and a.dtype == object):
            same = (b.isna() & a.isna()) | (b.astype(object) == a.astype(object))
            n = int((~same).sum())
            if n:
                chg.append((nm, c, n, b[~same].iloc[0], a[~same].iloc[0]))
display(pd.DataFrame(chg, columns=["dataset", "column", "rows changed", "example before", "example after"])
        if chg else pd.DataFrame({"result": ["source text values were already clean: no case/whitespace changes were necessary"]}))

print("3) Conformed identifiers (pattern-validated) and genuine NULLs preserved:")
idrows = []
for nm, frame in after.items():
    for c in ID_PATTERNS:
        if c in frame.columns:
            idrows.append((nm, c, int(frame[c].notna().sum()), int(frame[c].isna().sum()), frame[c].dropna().iloc[0]))
display(pd.DataFrame(idrows, columns=["dataset", "id column", "valid ids", "NULL ids", "example"]))
print("   NULL campaign in orders is NULL (not the string 'nan'):", orders.campaign.isna().sum(), "NULLs;",
      "'nan' strings:", int((orders.campaign == "nan").sum()))

print("\n4) Derived fields (computed here for display; computed again in SQL when the dw tables are loaded):")
derived = pd.DataFrame({
    "order_id": orders.order_id,
    "order_datetime": orders.order_datetime,
    "order_date_key": orders.order_datetime.dt.strftime("%Y%m%d").astype(int),
    "ship_date_key": orders.ship_datetime.dt.strftime("%Y%m%d").astype("Int64"),
    "net_revenue_check": (orders.subtotal_usd - orders.discount_amount_usd + orders.shipping_cost_usd + orders.tax_amount_usd).round(2),
    "order_total_usd": orders.order_total_usd,
})
display(derived.head(5))
ev_derived = pd.DataFrame({"event_id": events.event_id, "event_ts": events.event_ts,
                           "event_date_key": events.event_ts.dt.strftime("%Y%m%d").astype(int)})
display(ev_derived.head(3))
print("   order_total_usd == subtotal - discount + shipping + tax for",
      f"{(derived.net_revenue_check - derived.order_total_usd).abs().lt(0.02).mean():.1%} of orders")

---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [ ]:
# ========= Redshift Functions =========

session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)

def _rs_kwargs() -> Dict[str, Any]:
    """Build Redshift Data API connection parameters."""
    base = dict(Database=REDSHIFT_DATABASE)
    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP
        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN
    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:
        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER
    else:
        raise RuntimeError("Configure Redshift serverless OR provisioned for Data API.")
    return base

# Naming strategy: SQL in this notebook always uses the DDL's own names (stg.* and dw.*).
# Only if the Udacity user cannot CREATE SCHEMA, NAMING["public_fallback"] is switched on and EVERY statement is rewritten
# consistently to public.stg_* / public.dw_* (never a mix). Normally it stays False.
NAMING = {"public_fallback": False}
_AUTH_ERRORS = ("ExpiredTokenException", "ExpiredToken", "UnrecognizedClientException", "InvalidClientTokenId", "AccessDeniedException")
_TERMINAL = {"FINISHED", "FAILED", "ABORTED"}
LAST_STATEMENT: Dict[str, Any] = {}      # describe_statement() of the most recent statement (Duration is in nanoseconds)

def _apply_naming(sql: str) -> str:
    if NAMING["public_fallback"]:
        sql = re.sub(r"\bstg\.", "public.stg_", sql)
        sql = re.sub(r"\bdw\.", "public.dw_", sql)
    return sql

def _rs_value(field: Dict[str, Any], type_name: str):
    if field.get("isNull"):
        return None
    (kind, val), = field.items()
    if kind == "stringValue" and type_name in ("numeric", "decimal"):
        return float(Decimal(val))
    return val

def rs_exec(sql: str, return_results=False, timeout_s=900):
    """Execute SQL on Redshift via the Data API: submit, poll until FINISHED/FAILED/ABORTED, optionally fetch results.
    Returns a list of dicts when return_results=True, otherwise the statement id. Raises on failure or timeout."""
    sql = _apply_naming(sql)
    try:
        resp = rsd.execute_statement(Sql=sql, **_rs_kwargs())
    except botocore.exceptions.ClientError as e:
        if e.response["Error"]["Code"] in _AUTH_ERRORS:
            raise RuntimeError("AWS credentials are expired or invalid (lab sessions last ~4h). Refresh them from the Cloud Resources "
                               "tab and re-run the credentials cell - this is NOT a SQL problem.") from e
        raise
    stmt_id, t0, delay = resp["Id"], time.time(), 0.3
    while True:
        desc = rsd.describe_statement(Id=stmt_id)
        if desc["Status"] in _TERMINAL:
            break
        if time.time() - t0 > timeout_s:
            try:
                rsd.cancel_statement(Id=stmt_id)
            except Exception:
                pass
            raise TimeoutError(f"Redshift statement {stmt_id} still {desc['Status']} after {timeout_s}s: {sql[:200]!r}")
        time.sleep(delay)
        delay = min(delay * 1.5, 3.0)
    LAST_STATEMENT.clear()
    LAST_STATEMENT.update(desc)
    if desc["Status"] != "FINISHED":
        raise RuntimeError(f"Redshift statement {desc['Status']}: {desc.get('Error')}\n--- SQL ---\n{sql[:800]}")
    if not return_results:
        return stmt_id
    if not desc.get("HasResultSet"):
        return []
    rows, token = [], None
    while True:
        kw = dict(Id=stmt_id)
        if token:
            kw["NextToken"] = token
        page = rsd.get_statement_result(**kw)
        meta = page["ColumnMetadata"]
        names = [m.get("label") or m["name"] for m in meta]
        types = [m.get("typeName", "") for m in meta]
        for rec in page["Records"]:
            rows.append({n: _rs_value(f, t) for n, f, t in zip(names, rec, types)})
        token = page.get("NextToken")
        if not token:
            return rows

def rs_df(sql: str) -> pd.DataFrame:
    """Run a query and return the result as a DataFrame."""
    return pd.DataFrame(rs_exec(sql, return_results=True))

def rs_scalar(sql: str):
    rows = rs_exec(sql, return_results=True)
    return next(iter(rows[0].values())) if rows else None

_MAX_SQL_CHARS = 90_000      # Data API limit is 100 KB per statement

def _sql_literal(v, kind: str) -> str:
    v = _py(v)
    if v is None:
        return "NULL"
    if kind == "s":
        return "'" + str(v).replace("\x00", "").replace("\\", "\\\\").replace("'", "''") + "'"
    if kind == "ts":
        return "'" + pd.Timestamp(v).strftime("%Y-%m-%d %H:%M:%S") + "'"
    if kind == "i":
        return str(int(v))
    if kind == "f":
        return repr(float(v))
    if kind == "b":
        return "TRUE" if bool(v) else "FALSE"
    raise ValueError(kind)

def rs_batch_insert(table: str, colspec: List[Tuple[str, str]], df: pd.DataFrame, truncate: bool = True) -> int:
    """Load a DataFrame into a Redshift table with multi-row INSERTs (batched by row count AND statement size).
    truncate=True empties the staging table first so re-runs never duplicate rows. Returns rows sent."""
    cols = [c for c, _ in colspec]
    kinds = [k for _, k in colspec]
    head = f"INSERT INTO {table} ({', '.join(chr(34) + c + chr(34) for c in cols)}) VALUES "
    if truncate:
        rs_exec(f"TRUNCATE TABLE {table}")
    records = df_records(df, cols)
    batches, cur, size = [], [], len(head)
    for rec in records:
        tup = "(" + ", ".join(_sql_literal(v, k) for v, k in zip(rec, kinds)) + ")"
        if len(head) + len(tup) > _MAX_SQL_CHARS:        # a single row that can never fit: fail here, not as an opaque Data API error
            raise ValueError(f"{table}: one row needs {len(head) + len(tup):,} characters, more than the {_MAX_SQL_CHARS:,} allowed per statement")
        if cur and (len(cur) >= BATCH_SIZE or size + len(tup) > _MAX_SQL_CHARS):
            batches.append(cur); cur, size = [], len(head)
        cur.append(tup); size += len(tup) + 2
    if cur:
        batches.append(cur)
    it = tqdm(batches, desc=table) if TQDM else batches
    for b in it:
        rs_exec(head + ", ".join(b))
        if not TQDM:
            print(f"   {table}: inserted batch of {len(b)} rows")
    return len(records)

# ---- metadata helpers: use pg_catalog (readable by every user), NOT svv_table_info (permission denied in the lab) ----
_DIST_STYLE = {0: "EVEN", 1: "KEY", 8: "ALL", 10: "AUTO(ALL)", 11: "AUTO(EVEN)", 12: "AUTO(KEY)"}

def _logical(schema: str, table: str) -> str:
    """physical (schema, table) -> logical stg.x / dw.x name, whichever naming mode is active."""
    m = re.match(r"^(stg|dw)_(.+)$", table) if schema == "public" else None
    return f"{m.group(1)}.{m.group(2)}" if m else f"{schema}.{table}"

def rs_table_meta() -> pd.DataFrame:
    """One row per project table: logical name, diststyle (KEY(col)/ALL/EVEN), first sort key column."""
    df = rs_df("""
        SELECT n.nspname AS schema_name, c.relname AS table_name, c.reldiststyle AS ds,
               dk.attname AS distkey, sk.attname AS sortkey1
        FROM pg_class c
        JOIN pg_namespace n ON n.oid = c.relnamespace
        LEFT JOIN pg_attribute dk ON dk.attrelid = c.oid AND dk.attisdistkey = true AND dk.attnum > 0
        LEFT JOIN pg_attribute sk ON sk.attrelid = c.oid AND sk.attsortkeyord = 1
        WHERE c.relkind = 'r' AND n.nspname IN ('stg', 'dw', 'public')
    """)
    df["logical"] = [_logical(s, t) for s, t in zip(df.schema_name, df.table_name)]
    df = df[df.logical.isin([f"{r.schema}.{r.table}" for _, r in DDL_TABLES.iterrows()])].copy()
    df["diststyle"] = [f"KEY({k})" if d == 1 and k else _DIST_STYLE.get(int(d), str(d)) for d, k in zip(df.ds, df.distkey)]
    return df[["logical", "schema_name", "table_name", "diststyle", "sortkey1"]].sort_values("logical").reset_index(drop=True)

def rs_table_columns() -> Dict[str, set]:
    """logical table name -> set of column names, from pg_attribute."""
    df = rs_df("""
        SELECT n.nspname AS schema_name, c.relname AS table_name, a.attname AS column_name
        FROM pg_attribute a
        JOIN pg_class c ON c.oid = a.attrelid
        JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE a.attnum > 0 AND a.attisdropped = false AND c.relkind = 'r' AND n.nspname IN ('stg', 'dw', 'public')
    """)
    out: Dict[str, set] = {}
    for s, t, col in zip(df.schema_name, df.table_name, df.column_name):
        out.setdefault(_logical(s, t), set()).add(col)
    return out

print("Redshift functions defined: rs_exec(), rs_df(), rs_scalar(), rs_batch_insert(), rs_table_meta(), rs_table_columns()")

In [ ]:
# ========= Pipeline safety helpers: execution modes, naming detection, rebuild guard =========
# Pure functions (no database access): they are unit-tested in tests/ without AWS.
PIPELINE_MODES = ("INITIALIZE", "LOAD", "REBUILD")
_DDL_STMT = re.compile(r"^(DROP TABLE IF EXISTS|CREATE TABLE|CREATE OR REPLACE VIEW|CREATE SCHEMA IF NOT EXISTS)\s+(\w+)(?:\.(\w+))?", re.I)

def project_tables(ddl_tables: pd.DataFrame) -> set:
    """Logical names ('stg.orders_raw', 'dw.fact_orders', ...) of the 18 project tables defined by the DDL."""
    return {f"{r.schema}.{r.table}" for _, r in ddl_tables.iterrows()}

def check_rebuild_allowed(mode: str, database: str, allowed_databases, confirm: str) -> None:
    """Fail-closed guard. REBUILD (drop + re-create) needs BOTH an allow-listed database name AND an exact confirmation string."""
    if mode not in PIPELINE_MODES:
        raise ValueError(f"PIPELINE_MODE must be one of {PIPELINE_MODES}, got {mode!r}")
    if mode != "REBUILD":
        return
    if database not in allowed_databases:
        raise PermissionError(f"REBUILD refused: database {database!r} is not in REBUILD_ALLOWED_DATABASES {sorted(allowed_databases)}")
    if confirm != f"REBUILD {database}":
        raise PermissionError(f"REBUILD refused: set REBUILD_CONFIRM to exactly 'REBUILD {database}' to confirm that every project table is dropped")

def detect_naming(physical, project: set):
    """physical = iterable of (schemaname, tablename) found in the database.
    Returns 'schemas' (stg/dw schemas), 'public_fallback' (public.stg_*/public.dw_*) or None (no project table exists yet).
    Both conventions present at once is an error: the two must never be mixed."""
    in_schemas = {f"{s}.{t}" for s, t in physical if s in ("stg", "dw") and f"{s}.{t}" in project}
    in_public = {_logical(s, t) for s, t in physical if s == "public" and _logical(s, t) in project}
    if in_schemas and in_public:
        raise RuntimeError("project tables exist under BOTH naming conventions (stg/dw schemas and public.stg_*/dw_*); clean up one of them")
    return "schemas" if in_schemas else "public_fallback" if in_public else None

def existing_project_tables(physical, project: set) -> set:
    return {_logical(s, t) for s, t in physical if _logical(s, t) in project}

def plan_ddl(statements: List[str], mode: str, existing: set) -> List[str]:
    """Which DDL statements to run, given the mode and the project tables that already exist (logical names).
    INITIALIZE: skip every DROP, create only missing tables, keep populated tables untouched.
    LOAD: nothing is executed, but every project table must already exist.
    REBUILD: everything in file order (drops + creates). The caller must have passed check_rebuild_allowed() first."""
    if mode not in PIPELINE_MODES:
        raise ValueError(f"PIPELINE_MODE must be one of {PIPELINE_MODES}, got {mode!r}")
    if mode == "REBUILD":
        return list(statements)
    creates = []
    for s in statements:
        m = _DDL_STMT.match(s)
        if m and m.group(1).upper() == "CREATE TABLE":
            creates.append(f"{m.group(2)}.{m.group(3)}")
    if mode == "LOAD":
        missing = sorted(set(creates) - existing)
        if missing:
            raise RuntimeError(f"PIPELINE_MODE=LOAD but {len(missing)} project table(s) do not exist yet (e.g. {missing[:3]}); run once with INITIALIZE")
        return []
    plan = []
    for s in statements:
        m = _DDL_STMT.match(s)
        kind = m.group(1).upper() if m else ""
        if kind == "DROP TABLE IF EXISTS":
            continue                                   # INITIALIZE never drops
        if kind == "CREATE TABLE" and f"{m.group(2)}.{m.group(3)}" in existing:
            continue                                   # keep the populated table
        plan.append(s)
    return plan

def touched_objects(statements: List[str]) -> set:
    """Logical names of every table/view a statement list creates or drops (to prove nothing outside the project is touched)."""
    out = set()
    for s in statements:
        m = _DDL_STMT.match(s)
        if m and m.group(3):
            out.add(f"{m.group(2)}.{m.group(3)}")
    return out

def execute_warehouse_ddl(statements: List[str], ddl_tables: pd.DataFrame, mode: str, allowed_databases, confirm: str, say=print) -> int:
    """Create / keep / rebuild the project tables according to `mode`. Returns the number of statements executed.
    Decides the naming mode from what already exists in the database (never mixing the two conventions); on an empty database it tries
    CREATE SCHEMA and falls back to public.stg_* / public.dw_* when the user is not allowed to create schemas."""
    # safety first: only the project schemas stg / dw may ever be created or dropped
    for s in statements:
        m = _DDL_STMT.match(s)
        assert m and m.group(2) in ("stg", "dw"), f"refusing to run statement outside stg/dw: {s[:80]!r}"
    say(f"   {len(statements)} statements checked: they only touch the project schemas stg and dw")
    db = rs_scalar("SELECT current_database()")
    say(f"   Connected to Redshift database: {db}")
    check_rebuild_allowed(mode, db, allowed_databases, confirm)

    project = project_tables(ddl_tables)
    phys = rs_df("SELECT schemaname, tablename FROM pg_tables WHERE schemaname IN ('stg','dw','public')")
    phys_rows = list(zip(phys.schemaname, phys.tablename)) if len(phys) else []
    naming_found = detect_naming(phys_rows, project)
    if naming_found:
        NAMING["public_fallback"] = (naming_found == "public_fallback")      # the database already decides the naming mode
    existing = existing_project_tables(phys_rows, project)
    say(f"   Project tables that already exist: {len(existing)} of {len(project)}")

    plan = plan_ddl(statements, mode, existing)
    say(f"   Statements to execute in {mode} mode: {len(plan)} (of {len(statements)} in the DDL file)")
    if mode == "INITIALIZE" and existing:
        say("   INITIALIZE never drops: the tables that already exist are kept with their data.")

    done = 0
    schema_stmts = [s for s in plan if s.upper().startswith("CREATE SCHEMA")]
    other_stmts = [s for s in plan if not s.upper().startswith("CREATE SCHEMA")]
    if schema_stmts and not NAMING["public_fallback"]:
        try:
            for s in schema_stmts:
                rs_exec(s); done += 1
        except RuntimeError as e:
            if "permission denied" in str(e).lower() and naming_found is None:
                NAMING["public_fallback"] = True
                say("   ⚠ CREATE SCHEMA not permitted -> using the consistent public.stg_* / public.dw_* mapping for ALL statements")
            else:
                raise
    if mode == "REBUILD":
        # the materialized view built in Task 5 depends on fact_orders: drop it first so the tables can be dropped
        rs_exec("DROP MATERIALIZED VIEW IF EXISTS dw.mv_daily_revenue")
    for s in other_stmts:
        rs_exec(s); done += 1
    return done


## Task 4.2: Execute DDL and Create Tables

In [ ]:
print("="*60)
print("TASK 4: Loading Data into Redshift")
print("="*60)

print(f"\n📋 Step 1: Warehouse DDL (PIPELINE_MODE = {PIPELINE_MODE})...")
with stage("execute DDL", rows_in=len(DDL_STATEMENTS)) as st:
    done = execute_warehouse_ddl(DDL_STATEMENTS, DDL_TABLES, PIPELINE_MODE, REBUILD_ALLOWED_DATABASES, REBUILD_CONFIRM)
    st.set_out(done)
print(f"   Executed {done} DDL statements (naming mode: {'public_fallback' if NAMING['public_fallback'] else 'stg/dw schemas'})")


In [ ]:
# Verify the DDL really took effect by querying Redshift's own catalog (not by trusting the success message)
print("📋 Verifying created objects in Redshift metadata...")
meta = rs_table_meta()
display(meta)
got = rs_table_columns()
problems = []
for _, r in DDL_TABLES.iterrows():
    name = f"{r.schema}.{r.table}"
    if name not in got:
        problems.append(f"missing table {name}")
    elif set(r.columns) - got[name]:
        problems.append(f"{name}: missing columns {sorted(set(r.columns) - got[name])}")
print("Tables expected from DDL:", len(DDL_TABLES), "| tables found with all expected columns:", len(DDL_TABLES) - len(problems))
print("Naming mode:", "public.stg_*/public.dw_* (CREATE SCHEMA was not permitted)" if NAMING["public_fallback"] else "stg / dw schemas")
assert not problems, problems
# distribution / sort keys stored by Redshift must equal what the DDL declares
exp = DDL_TABLES.set_index(DDL_TABLES.schema + "." + DDL_TABLES.table)
m2 = meta.set_index("logical")
bad = []
for name, r in exp.iterrows():
    want_dist = f"KEY({r.distkey})" if r.diststyle == "KEY" else r.diststyle
    got_dist = m2.loc[name, "diststyle"]
    if want_dist != "AUTO/EVEN" and want_dist != got_dist:
        bad.append(f"{name}: DDL says {want_dist}, Redshift has {got_dist}")
    if r.sortkey and r.sortkey != m2.loc[name, "sortkey1"]:
        bad.append(f"{name}: DDL sortkey {r.sortkey}, Redshift has {m2.loc[name, 'sortkey1']}")
print("Distribution/sort keys differing from the DDL:", bad or "none")
assert not bad, bad
print("✅ Staging, dimension and fact tables exist with all expected columns, distribution and sort keys as designed")

## Task 4.3: Load Staging Tables

In [ ]:
print("\n📦 Step 2: Loading staging tables...")

STAGING_LOADS = [("stg.orders_raw", ORDERS_COLSPEC, orders), ("stg.events_raw", EVENTS_COLSPEC, events), ("stg.edges_raw", EDGES_COLSPEC, edges)]
staging_counts = {}
for tbl, spec, frame in STAGING_LOADS:
    print(f"\n  Loading {frame.shape[0]:,} rows into {tbl}...")
    with stage(f"load {tbl}", rows_in=len(frame)) as st:
        rs_batch_insert(tbl, spec, frame)
        n = int(rs_scalar(f"SELECT COUNT(*) FROM {tbl}"))
        st.set_out(n)
        staging_counts[tbl] = n
    assert n == len(frame), f"{tbl}: {n} rows in Redshift but {len(frame)} sent"
    print(f"  {tbl}: {n:,} rows in Redshift == {len(frame):,} transformed rows ✔")
print("\n✅ Staging tables loaded and counted in Redshift!")

## Task 4.4: Populate Dimension Tables

In [ ]:
# ========= Warehouse SQL (definitions only; executed in the next cells) =========
# Update semantics, stated precisely:
#   * every statement is idempotent: running it twice on unchanged staging data changes nothing;
#   * NEW natural keys are inserted;
#   * a CHANGED attribute of an existing key is overwritten in place (Type 1). No history is kept, even though the
#     customer/product dimensions carry SCD-2-ready columns (effective_from / effective_to / is_current);
#   * keys that disappeared from the source are NOT deleted (see STALE_FACT_CHECKS: the load stops with an explicit error).
def _ne(a: str, b: str) -> str:
    """NULL-safe 'a differs from b' without IS DISTINCT FROM (plain portable SQL)."""
    return f"({a} <> {b} OR ({a} IS NULL AND {b} IS NOT NULL) OR ({a} IS NOT NULL AND {b} IS NULL))"

SQL_DIM_DATE = """
INSERT INTO dw.dim_date (date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend)
SELECT
    CAST(to_char(dt, 'YYYYMMDD') AS INTEGER),
    dt,
    EXTRACT(YEAR FROM dt)::SMALLINT, EXTRACT(QUARTER FROM dt)::SMALLINT, EXTRACT(MONTH FROM dt)::SMALLINT,
    EXTRACT(DAY FROM dt)::SMALLINT, EXTRACT(WEEK FROM dt)::SMALLINT, EXTRACT(DOW FROM dt)::SMALLINT,
    (EXTRACT(DOW FROM dt) IN (0, 6))::BOOLEAN
FROM (
    SELECT order_datetime::DATE AS dt FROM stg.orders_raw WHERE order_datetime IS NOT NULL
    UNION SELECT ship_datetime::DATE FROM stg.orders_raw WHERE ship_datetime IS NOT NULL
    UNION SELECT event_ts::DATE FROM stg.events_raw WHERE event_ts IS NOT NULL
    UNION SELECT "timestamp"::DATE FROM stg.edges_raw WHERE "timestamp" IS NOT NULL
) d
WHERE CAST(to_char(dt, 'YYYYMMDD') AS INTEGER) NOT IN (SELECT date_key FROM dw.dim_date)
"""

# Attribute selection rule (deterministic): the most recent record wins; when two records carry the same timestamp, the
# record with the HIGHER record id (order_id / event_id / edge_id, string comparison) wins. The tie-break has no business
# meaning, it only makes the result reproducible. Country and state always come from ONE record (never mixed).
CUSTOMER_CTES = """
ids AS (
    SELECT customer_id AS cid FROM stg.orders_raw WHERE customer_id IS NOT NULL
    UNION SELECT customer_id FROM stg.events_raw WHERE customer_id IS NOT NULL
    UNION SELECT from_node_id FROM stg.edges_raw WHERE from_node_type = 'Customer' AND from_node_id IS NOT NULL
    UNION SELECT to_node_id FROM stg.edges_raw WHERE to_node_type = 'Customer' AND to_node_id IS NOT NULL
),
geo AS (            -- geography: orders first (src_rank 1), else events; most recent record wins, ties -> higher record id
    SELECT cid, country, state FROM (
        SELECT cid, country, state, ROW_NUMBER() OVER (PARTITION BY cid ORDER BY src_rank, ts DESC, rid DESC) AS rn
        FROM (
            SELECT customer_id AS cid, country, state, order_datetime AS ts, 1 AS src_rank, order_id AS rid FROM stg.orders_raw
            UNION ALL
            SELECT customer_id, country, state, event_ts, 2, event_id FROM stg.events_raw
        ) u WHERE cid IS NOT NULL AND country IS NOT NULL
    ) g WHERE rn = 1
),
seg AS (            -- segment only exists on graph edges leaving a Customer node; ties -> higher edge_id
    SELECT cid, customer_segment FROM (
        SELECT from_node_id AS cid, customer_segment,
               ROW_NUMBER() OVER (PARTITION BY from_node_id ORDER BY "timestamp" DESC, edge_id DESC) AS rn
        FROM stg.edges_raw WHERE from_node_type = 'Customer' AND customer_segment IS NOT NULL
    ) s WHERE rn = 1
),
login AS (          -- latest event wins; ties -> higher event_id
    SELECT cid, is_logged_in FROM (
        SELECT customer_id AS cid, is_logged_in,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY event_ts DESC, event_id DESC) AS rn
        FROM stg.events_raw WHERE is_logged_in IS NOT NULL
    ) l WHERE rn = 1
)
"""
CUSTOMER_FROM = """
FROM ids
LEFT JOIN geo   ON geo.cid   = ids.cid
LEFT JOIN seg   ON seg.cid   = ids.cid
LEFT JOIN login ON login.cid = ids.cid
"""
SQL_DIM_CUSTOMER = f"""
INSERT INTO dw.dim_customer (customer_id, country, state, customer_segment, is_logged_in, effective_from, effective_to, is_current)
WITH {CUSTOMER_CTES}
SELECT ids.cid, geo.country, geo.state, seg.customer_segment, login.is_logged_in,
       CURRENT_TIMESTAMP, CAST(NULL AS TIMESTAMP), TRUE
{CUSTOMER_FROM}
WHERE ids.cid NOT IN (SELECT customer_id FROM dw.dim_customer WHERE customer_id IS NOT NULL)
"""
_CUST_ATTRS = ["country", "state", "customer_segment", "is_logged_in"]
SQL_DIM_CUSTOMER_UPDATE = f"""
WITH {CUSTOMER_CTES},
src AS (SELECT ids.cid AS customer_id, geo.country, geo.state, seg.customer_segment, login.is_logged_in {CUSTOMER_FROM})
UPDATE dw.dim_customer
SET {", ".join(f"{c} = src.{c}" for c in _CUST_ATTRS)}
FROM src
WHERE dw.dim_customer.customer_id = src.customer_id AND dw.dim_customer.is_current = TRUE
  AND ({" OR ".join(_ne(f"dw.dim_customer.{c}", f"src.{c}") for c in _CUST_ATTRS)})
"""

PRODUCT_CTES = """
ids AS (
    SELECT product_id AS pid FROM stg.events_raw WHERE product_id IS NOT NULL
    UNION SELECT product_id FROM stg.edges_raw WHERE product_id IS NOT NULL
    UNION SELECT from_node_id FROM stg.edges_raw WHERE from_node_type = 'Product' AND from_node_id IS NOT NULL
    UNION SELECT to_node_id FROM stg.edges_raw WHERE to_node_type = 'Product' AND to_node_id IS NOT NULL
),
cat AS (            -- category: graph first (src_rank 1), else events; most recent record wins, ties -> higher record id
    SELECT pid, category FROM (
        SELECT pid, category, ROW_NUMBER() OVER (PARTITION BY pid ORDER BY src_rank, ts DESC, rid DESC) AS rn
        FROM (
            SELECT product_id AS pid, category, "timestamp" AS ts, 1 AS src_rank, edge_id AS rid FROM stg.edges_raw
            UNION ALL
            SELECT product_id, category, event_ts, 2, event_id FROM stg.events_raw
        ) u WHERE pid IS NOT NULL AND category IS NOT NULL
    ) c WHERE rn = 1
),
px AS (             -- latest observed price and price bucket from the graph (one record); ties -> higher edge_id
    SELECT pid, price_bucket, unit_price_usd FROM (
        SELECT product_id AS pid, price_bucket, unit_price_usd,
               ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY "timestamp" DESC, edge_id DESC) AS rn
        FROM stg.edges_raw WHERE product_id IS NOT NULL AND unit_price_usd IS NOT NULL
    ) p WHERE rn = 1
)
"""
PRODUCT_FROM = """
FROM ids
LEFT JOIN cat ON cat.pid = ids.pid
LEFT JOIN px  ON px.pid  = ids.pid
"""
SQL_DIM_PRODUCT = f"""
INSERT INTO dw.dim_product (product_id, category, price_bucket, current_unit_price_usd, effective_from, effective_to, is_current)
WITH {PRODUCT_CTES}
SELECT ids.pid, cat.category, px.price_bucket, px.unit_price_usd, CURRENT_TIMESTAMP, CAST(NULL AS TIMESTAMP), TRUE
{PRODUCT_FROM}
WHERE ids.pid NOT IN (SELECT product_id FROM dw.dim_product WHERE product_id IS NOT NULL)
"""
_PROD_ATTRS = [("category", "category"), ("price_bucket", "price_bucket"), ("current_unit_price_usd", "unit_price_usd")]
SQL_DIM_PRODUCT_UPDATE = f"""
WITH {PRODUCT_CTES},
src AS (SELECT ids.pid AS product_id, cat.category, px.price_bucket, px.unit_price_usd AS current_unit_price_usd {PRODUCT_FROM})
UPDATE dw.dim_product
SET {", ".join(f"{c} = src.{c}" for c, _ in _PROD_ATTRS)}
FROM src
WHERE dw.dim_product.product_id = src.product_id AND dw.dim_product.is_current = TRUE
  AND ({" OR ".join(_ne(f"dw.dim_product.{c}", f"src.{c}") for c, _ in _PROD_ATTRS)})
"""

# (dimension table, natural-key column, [staging selects feeding it]) - these dimensions carry no attributes to update
SIMPLE_DIMS = [
    ("dim_channel",         "channel",         ["SELECT channel FROM stg.orders_raw"]),
    ("dim_device",          "device_type",     ["SELECT device_type FROM stg.orders_raw", "SELECT device_type FROM stg.events_raw"]),
    ("dim_browser",         "browser",         ["SELECT browser FROM stg.orders_raw", "SELECT browser FROM stg.events_raw"]),
    ("dim_campaign",        "campaign",        ["SELECT campaign FROM stg.orders_raw", "SELECT campaign FROM stg.edges_raw"]),
    ("dim_payment_method",  "payment_method",  ["SELECT payment_method FROM stg.orders_raw"]),
    ("dim_shipping_method", "shipping_method", ["SELECT shipping_method FROM stg.orders_raw"]),
    ("dim_os",              "os",              ["SELECT os FROM stg.events_raw"]),
    ("dim_referrer",        "referrer",        ["SELECT referrer FROM stg.events_raw"]),
    ("dim_ab_variant",      "ab_variant",      ["SELECT ab_variant FROM stg.events_raw"]),
]
def sql_simple_dim(table: str, col: str, selects: List[str]) -> str:
    union = " UNION ".join(selects)
    return (f"INSERT INTO dw.{table} ({col}) SELECT DISTINCT {col} FROM ({union}) x "
            f"WHERE {col} IS NOT NULL AND {col} NOT IN (SELECT {col} FROM dw.{table} WHERE {col} IS NOT NULL)")

def dimension_statements() -> List[Tuple[str, str]]:
    """(label, sql) in execution order: dim_date, customer (insert + update), product (insert + update), the nine small dimensions."""
    out = [("dim_date", SQL_DIM_DATE), ("dim_customer: insert new", SQL_DIM_CUSTOMER), ("dim_customer: update changed", SQL_DIM_CUSTOMER_UPDATE),
           ("dim_product: insert new", SQL_DIM_PRODUCT), ("dim_product: update changed", SQL_DIM_PRODUCT_UPDATE)]
    return out + [(t, sql_simple_dim(t, c, sel)) for t, c, sel in SIMPLE_DIMS]

# ---- facts: one SELECT per fact whose output aliases are the fact column names; used for both the INSERT and the UPDATE ----
# Each fact row is built from exactly ONE staging row. Dimension lookups are LEFT JOINs on UNIQUE natural keys, so rows are
# neither multiplied nor dropped; a missing optional key simply yields a NULL surrogate key.
FACT_ORDERS_COLS = ["order_id", "customer_sk", "order_date_key", "ship_date_key", "channel_sk", "device_sk", "browser_sk", "campaign_sk",
    "payment_method_sk", "shipping_method_sk", "primary_category", "num_distinct_items", "subtotal_usd", "discount_rate",
    "discount_amount_usd", "shipping_cost_usd", "tax_rate", "tax_amount_usd", "order_total_usd", "order_weight_kg",
    "delivery_days", "on_time_delivery", "authorization_approved", "returned"]
FACT_ORDERS_SELECT = """
SELECT o.order_id, dc.customer_sk,
       CAST(to_char(o.order_datetime, 'YYYYMMDD') AS INTEGER) AS order_date_key, CAST(to_char(o.ship_datetime, 'YYYYMMDD') AS INTEGER) AS ship_date_key,
       ch.channel_sk, dv.device_sk, br.browser_sk, cp.campaign_sk, pm.payment_method_sk, sm.shipping_method_sk,
       o.primary_category, o.num_distinct_items, o.subtotal_usd, o.discount_rate, o.discount_amount_usd, o.shipping_cost_usd,
       o.tax_rate, o.tax_amount_usd, o.order_total_usd, o.order_weight_kg, o.delivery_days, o.on_time_delivery,
       o.authorization_approved, o.returned
FROM stg.orders_raw o
LEFT JOIN dw.dim_customer        dc ON dc.customer_id = o.customer_id AND dc.is_current = TRUE
LEFT JOIN dw.dim_channel         ch ON ch.channel = o.channel
LEFT JOIN dw.dim_device          dv ON dv.device_type = o.device_type
LEFT JOIN dw.dim_browser         br ON br.browser = o.browser
LEFT JOIN dw.dim_campaign        cp ON cp.campaign = o.campaign
LEFT JOIN dw.dim_payment_method  pm ON pm.payment_method = o.payment_method
LEFT JOIN dw.dim_shipping_method sm ON sm.shipping_method = o.shipping_method
"""
FACT_EVENTS_COLS = ["event_id", "customer_sk", "product_sk", "event_date_key", "session_id", "event_type", "device_sk", "browser_sk",
    "os_sk", "referrer_sk", "ab_variant_sk", "page_depth", "latency_ms", "dwell_seconds", "cart_value_usd", "discount_rate", "fraud_score",
    "payment_outcome", "sequence_num", "category", "promo_code"]
FACT_EVENTS_SELECT = """
SELECT e.event_id, dc.customer_sk, dp.product_sk, CAST(to_char(e.event_ts, 'YYYYMMDD') AS INTEGER) AS event_date_key, e.session_id, e.event_type,
       dv.device_sk, br.browser_sk, os.os_sk, rf.referrer_sk, ab.ab_variant_sk,
       e.page_depth, e.latency_ms, e.dwell_seconds, e.cart_value_usd, e.discount_rate, e.fraud_score,
       e.payment_outcome, e.sequence_num, e.category, e.promo_code
FROM stg.events_raw e
LEFT JOIN dw.dim_customer   dc ON dc.customer_id = e.customer_id AND dc.is_current = TRUE
LEFT JOIN dw.dim_product    dp ON dp.product_id  = e.product_id  AND dp.is_current = TRUE
LEFT JOIN dw.dim_device     dv ON dv.device_type = e.device_type
LEFT JOIN dw.dim_browser    br ON br.browser     = e.browser
LEFT JOIN dw.dim_os         os ON os.os          = e.os
LEFT JOIN dw.dim_referrer   rf ON rf.referrer    = e.referrer
LEFT JOIN dw.dim_ab_variant ab ON ab.ab_variant  = e.ab_variant
"""
FACT_EDGES_COLS = ["edge_id", "event_date_key", "relationship", "from_customer_sk", "to_customer_sk", "from_product_sk",
    "to_product_sk", "order_id", "category", "campaign_sk", "customer_segment", "region", "state", "edge_strength", "price_bucket",
    "prior_interactions", "dwell_seconds", "unit_price_usd", "quantity", "returned_flag", "auth_approved"]
# Graph nodes are routed by node TYPE: a Customer node only ever joins dim_customer, a Product node only dim_product.
FACT_EDGES_SELECT = """
SELECT g.edge_id, CAST(to_char(g."timestamp", 'YYYYMMDD') AS INTEGER) AS event_date_key, g.relationship,
       fc.customer_sk AS from_customer_sk, tc.customer_sk AS to_customer_sk, fp.product_sk AS from_product_sk, tp.product_sk AS to_product_sk,
       g.order_id, g.category, cp.campaign_sk, g.customer_segment, g.region, g.state, g.edge_strength, g.price_bucket,
       g.prior_interactions, g.dwell_seconds, g.unit_price_usd, g.quantity, g.returned_flag, g.auth_approved
FROM stg.edges_raw g
LEFT JOIN dw.dim_customer fc ON g.from_node_type = 'Customer' AND fc.customer_id = g.from_node_id AND fc.is_current = TRUE
LEFT JOIN dw.dim_customer tc ON g.to_node_type   = 'Customer' AND tc.customer_id = g.to_node_id   AND tc.is_current = TRUE
LEFT JOIN dw.dim_product  fp ON g.from_node_type = 'Product'  AND fp.product_id  = g.from_node_id AND fp.is_current = TRUE
LEFT JOIN dw.dim_product  tp ON g.to_node_type   = 'Product'  AND tp.product_id  = g.to_node_id   AND tp.is_current = TRUE
LEFT JOIN dw.dim_campaign cp ON cp.campaign = g.campaign
"""

def sql_fact_insert(fact: str, key: str, cols: List[str], select_sql: str) -> str:
    """Insert the staging rows whose natural key is not in the fact yet."""
    return (f"INSERT INTO dw.{fact} ({', '.join(cols)}) SELECT {', '.join('s.' + c for c in cols)} FROM ({select_sql}) s "
            f"WHERE NOT EXISTS (SELECT 1 FROM dw.{fact} f WHERE f.{key} = s.{key})")

def sql_fact_update(fact: str, key: str, cols: List[str], select_sql: str) -> str:
    """Apply corrections: overwrite the columns of an existing fact row whose staging values differ (NULL-safe). Unchanged rows are not written."""
    value_cols = [c for c in cols if c != key]
    diff = " OR ".join(_ne(f"dw.{fact}.{c}", f"s.{c}") for c in value_cols)
    return (f"UPDATE dw.{fact} SET {', '.join(f'{c} = s.{c}' for c in value_cols)} FROM ({select_sql}) s "
            f"WHERE dw.{fact}.{key} = s.{key} AND ({diff})")

FACTS = [("fact_orders", "order_id", FACT_ORDERS_COLS, FACT_ORDERS_SELECT, "stg.orders_raw"),
         ("fact_events", "event_id", FACT_EVENTS_COLS, FACT_EVENTS_SELECT, "stg.events_raw"),
         ("fact_graph_edges", "edge_id", FACT_EDGES_COLS, FACT_EDGES_SELECT, "stg.edges_raw")]

def fact_statements() -> List[Tuple[str, str]]:
    """(label, sql) in execution order: for each fact first the corrections, then the new rows."""
    out = []
    for fact, key, cols, sel, _ in FACTS:
        out.append((f"{fact}: update changed", sql_fact_update(fact, key, cols, sel)))
        out.append((f"{fact}: insert new", sql_fact_insert(fact, key, cols, sel)))
    return out

# Source records that were removed are NOT propagated (no delete logic). They are detected instead: a fact row without
# a staging row means the source lost a record, and the load stops with an explicit message.
STALE_FACT_CHECKS = [(fact, f"SELECT COUNT(*) FROM dw.{fact} f WHERE NOT EXISTS (SELECT 1 FROM {stg} s WHERE s.{key} = f.{key})")
                     for fact, key, _, _, stg in FACTS]

def check_staging_complete(expected: Dict[str, int]) -> None:
    """Fail closed: the facts may only be touched while staging holds exactly the transformed rows (an interrupted staging load must
    not update the warehouse from a partial copy). expected = {"stg.orders_raw": n_rows, ...}."""
    for tbl, n in expected.items():
        got = int(rs_scalar(f"SELECT COUNT(*) FROM {tbl}"))
        if got != n:
            raise RuntimeError(f"{tbl} holds {got} rows but {n} were transformed: staging is incomplete, facts NOT touched")

def check_no_stale_facts() -> None:
    """Source records that vanished from the source are not deleted by this pipeline; stop with an explicit message instead of silently keeping them."""
    stale = {fact: int(rs_scalar(sql)) for fact, sql in STALE_FACT_CHECKS}
    if any(stale.values()):
        raise RuntimeError(f"facts without a staging row (source records were removed; deletes are not supported): {stale}")


In [ ]:
print("\n📊 Step 3: Populating dimension tables...")
# Insert new natural keys and overwrite changed attributes (Type 1); running this twice on unchanged staging data changes nothing.

with stage("populate dimensions") as st:
    for label, sql in dimension_statements():
        rs_exec(sql); print(f"  ✓ {label}")
    dims = [r.table for _, r in DDL_TABLES.iterrows() if r.kind == "dimension"]
    union = " UNION ALL ".join(f"SELECT '{d}' AS table_name, COUNT(*) AS n FROM dw.{d}" for d in dims)
    dim_counts = rs_df(f"SELECT * FROM ({union}) t ORDER BY 1")
    st.set_out(int(dim_counts.n.sum()))
print("\nDimension row counts:")
display(dim_counts)
assert (dim_counts.n > 0).all(), "an empty dimension table"

# natural-key uniqueness (Redshift does not enforce PRIMARY KEY, so we check)
nk = [("dim_customer", "customer_id"), ("dim_product", "product_id"), ("dim_date", "date_key")] + [(t, c) for t, c, _ in SIMPLE_DIMS]
dups = {t: int(rs_scalar(f"SELECT COUNT(*) FROM (SELECT {c} FROM dw.{t} GROUP BY {c} HAVING COUNT(*) > 1) d")) for t, c in nk}
print("Natural-key duplicates per dimension (must all be 0):", dups)
assert not any(dups.values())
print("\n✅ All dimension tables populated (12 dimensions, no duplicate natural keys)!")


## Task 4.5: Populate Fact Tables

In [ ]:
print("\n📊 Step 4: Populating fact tables...")

check_staging_complete({"stg.orders_raw": len(orders), "stg.events_raw": len(events), "stg.edges_raw": len(edges)})
check_no_stale_facts()

for label, sql in fact_statements():
    with stage(f"populate {label}") as st:
        rs_exec(sql)
        if label.endswith("insert new"):
            st.set_out(int(rs_scalar(f"SELECT COUNT(*) FROM dw.{label.split(':')[0]}")))
    print(f"  ✓ {label}")

fact_counts = {t: int(rs_scalar(f"SELECT COUNT(*) FROM dw.{t}")) for t in ("fact_orders", "fact_events", "fact_graph_edges")}
print("\nFact row counts:", fact_counts)
assert fact_counts["fact_orders"] == staging_counts["stg.orders_raw"]
assert fact_counts["fact_events"] == staging_counts["stg.events_raw"]
assert fact_counts["fact_graph_edges"] == staging_counts["stg.edges_raw"]
print("\n✅ Fact tables populated; every staging row produced exactly one fact row")


In [ ]:
# ===== Row-count reconciliation across every stage: CSV -> source DB -> extracted -> transformed -> staging -> fact =====
print("RECONCILIATION")
print("="*60)
recon = pd.DataFrame([
    ("Orders",      len(orders_df), source_counts["PostgreSQL"], len(orders_extracted), len(orders), staging_counts["stg.orders_raw"], fact_counts["fact_orders"]),
    ("Events",      len(events_df), source_counts["Cassandra"],  len(events_extracted), len(events), staging_counts["stg.events_raw"], fact_counts["fact_events"]),
    ("Graph edges", len(edges_df),  source_counts["Neo4j"],      len(edges_extracted),  len(edges),  staging_counts["stg.edges_raw"],  fact_counts["fact_graph_edges"]),
], columns=["Dataset", "Source CSV", "Source DB", "Extracted", "Transformed", "Staging", "Final Fact"])
recon["Status"] = np.where(recon.iloc[:, 1:7].nunique(axis=1) == 1, "PASS (no record lost or added)", "FAIL")
display(recon)
assert (recon.Status.str.startswith("PASS")).all()

# value-level control totals: catch loss/duplication that a row count alone would miss
tot = pd.DataFrame([
    ("orders: SUM(order_total_usd)",  round(float(orders_df.order_total_usd.sum()), 2),
        float(rs_scalar("SELECT SUM(order_total_usd) FROM stg.orders_raw")), float(rs_scalar("SELECT SUM(order_total_usd) FROM dw.fact_orders"))),
    ("orders: COUNT(DISTINCT customer_id)", orders_df.customer_id.nunique(),
        int(rs_scalar("SELECT COUNT(DISTINCT customer_id) FROM stg.orders_raw")), int(rs_scalar("SELECT COUNT(DISTINCT customer_sk) FROM dw.fact_orders"))),
    ("events: SUM(cart_value_usd)",   round(float(events_df.cart_value_usd.sum()), 2),
        float(rs_scalar("SELECT SUM(cart_value_usd) FROM stg.events_raw")), float(rs_scalar("SELECT SUM(cart_value_usd) FROM dw.fact_events"))),
    ("events: COUNT(DISTINCT product_id)", events_df.product_id.nunique(),
        int(rs_scalar("SELECT COUNT(DISTINCT product_id) FROM stg.events_raw")), int(rs_scalar("SELECT COUNT(DISTINCT product_sk) FROM dw.fact_events"))),
    ("edges: SUM(unit_price_usd*quantity)", round(float((edges_df.unit_price_usd * edges_df.quantity).sum()), 2),
        float(rs_scalar("SELECT SUM(unit_price_usd*quantity) FROM stg.edges_raw")), float(rs_scalar("SELECT SUM(unit_price_usd*quantity) FROM dw.fact_graph_edges"))),
], columns=["control total", "CSV", "staging", "fact"])
tot["match"] = [(abs(a - b) < 0.01 and abs(a - c) < 0.01) for a, b, c in zip(tot.CSV, tot.staging, tot.fact)]
display(tot)
assert tot["match"].all(), "control totals differ"
print("✅ Counts and control totals reconcile end-to-end")

---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

In [ ]:
print("="*60)
print("TASK 5: Optimize Performance")
print("="*60)

# ---- 5.1 Verify distribution styles, sort keys and encodings that Redshift actually applied --------------------
print("\n🔍 Distribution and sort keys as stored in Redshift (pg_class / pg_attribute):")
phys = rs_table_meta()
display(phys)
dist_check = phys.set_index("logical")
exp = {"dw.fact_orders": "KEY(customer_sk)", "dw.fact_events": "KEY(customer_sk)", "dw.fact_graph_edges": "KEY(to_product_sk)",
       "dw.dim_customer": "KEY(customer_sk)", "dw.dim_product": "KEY(product_sk)", "dw.dim_date": "ALL", "dw.dim_channel": "ALL"}
for t, want in exp.items():
    assert dist_check.loc[t, "diststyle"] == want, f"{t}: expected {want}, Redshift has {dist_check.loc[t, 'diststyle']}"
print("✅ Distribution styles in Redshift match the design for the fact tables and key dimensions")

print("\n🔍 Column encodings (pg_table_def) for the fact tables:")
try:
    enc = rs_df("""
        SELECT schemaname, tablename, "column" AS col, encoding FROM pg_table_def
        WHERE schemaname IN ('dw', 'public') AND tablename LIKE '%fact%'
    """)
    enc["logical"] = [_logical(s, t) for s, t in zip(enc.schemaname, enc.tablename)]
    enc = enc[enc.logical.str.startswith("dw.fact_")]
    if enc.empty:
        print("   (pg_table_def shows only tables on the session search_path; none listed for this session)")
    else:
        display(enc.groupby(["logical", "encoding"]).size().rename("columns").reset_index())
except Exception as e:
    print("   encoding listing not available:", str(e)[:200])

# ---- 5.2 Materialized view for daily revenue ------------------------------------------------------------------
print("\n📊 Creating materialized view for daily revenue...")
with stage("create materialized view dw.mv_daily_revenue") as st:
    rs_exec("DROP MATERIALIZED VIEW IF EXISTS dw.mv_daily_revenue")
    rs_exec("""
        CREATE MATERIALIZED VIEW dw.mv_daily_revenue AS
        SELECT order_date_key,
               COUNT(*)             AS orders,
               SUM(order_total_usd) AS revenue_usd,
               AVG(order_total_usd) AS avg_order_value
        FROM dw.fact_orders
        GROUP BY order_date_key
    """)
    mv_rows = int(rs_scalar("SELECT COUNT(*) FROM dw.mv_daily_revenue"))
    st.set_out(mv_rows)
print(f"   dw.mv_daily_revenue created: {mv_rows:,} daily rows")
assert mv_rows > 0, "materialized view is empty"

# verify the view against an equivalent aggregation of the fact table (must be identical in both directions)
diff = int(rs_scalar("""
    SELECT COUNT(*) FROM (
        (SELECT order_date_key, orders, ROUND(revenue_usd, 2) FROM dw.mv_daily_revenue
         EXCEPT SELECT order_date_key, COUNT(*), ROUND(SUM(order_total_usd), 2) FROM dw.fact_orders GROUP BY order_date_key)
        UNION ALL
        (SELECT order_date_key, COUNT(*), ROUND(SUM(order_total_usd), 2) FROM dw.fact_orders GROUP BY order_date_key
         EXCEPT SELECT order_date_key, orders, ROUND(revenue_usd, 2) FROM dw.mv_daily_revenue)
    ) d
"""))
print(f"   Rows that differ between the MV and a fresh aggregation of fact_orders: {diff}")
assert diff == 0
mv_total = float(rs_scalar("SELECT SUM(revenue_usd) FROM dw.mv_daily_revenue"))
fact_total = float(rs_scalar("SELECT SUM(order_total_usd) FROM dw.fact_orders"))
print(f"   Total revenue: MV = {mv_total:,.2f}  fact_orders = {fact_total:,.2f}  (difference {abs(mv_total - fact_total):.4f})")
assert abs(mv_total - fact_total) < 0.01

# the view must be refreshable after new data is loaded
rs_exec("REFRESH MATERIALIZED VIEW dw.mv_daily_revenue")
print("   REFRESH MATERIALIZED VIEW dw.mv_daily_revenue executed successfully")
try:
    display(rs_df("SELECT name, is_stale, state FROM stv_mv_info WHERE name = 'mv_daily_revenue'"))
except Exception as e:
    print("   (stv_mv_info not readable:", str(e)[:120], ")")

In [ ]:
print("\n📊 Running ANALYZE on tables...")
analyze_tables = ["fact_orders", "fact_events", "fact_graph_edges", "dim_customer", "dim_product", "dim_date"]
with stage("ANALYZE", rows_in=len(analyze_tables)) as st:
    for table in analyze_tables:
        rs_exec(f"ANALYZE dw.{table}")
        print(f"  ✓ ANALYZE complete: {table}")
    st.set_out(len(analyze_tables))
try:
    display(rs_df("""SELECT 'fact_orders' AS table_name, COUNT(*) AS n FROM dw.fact_orders UNION ALL SELECT 'fact_events', COUNT(*) FROM dw.fact_events
                     UNION ALL SELECT 'fact_graph_edges', COUNT(*) FROM dw.fact_graph_edges"""))
except Exception as e:
    print("row counts unavailable:", str(e)[:120])

print("\n✅ Task 5 optimizations applied: distribution/sort keys verified, materialized view created and checked, ANALYZE run")

In [ ]:
# ===== Performance evidence: base-table aggregation vs materialized view, and join colocation in the query plans =====
print("PERFORMANCE MEASUREMENTS")
print("="*60)
RUNS = 7

def timed(sql_template: str, runs: int = RUNS):
    """Run a query several times and return per-run elapsed milliseconds as measured by Redshift (describe_statement.Duration).
    A different no-op predicate on every run (order_date_key <> -n) gives every run unique query text, so the result cache
    cannot answer it and every run really executes."""
    out = []
    for n in range(1, runs + 1):
        rs_exec(sql_template.format(salt=f"order_date_key <> -{n}"), return_results=True)
        out.append(LAST_STATEMENT["Duration"] / 1e6)          # ns -> ms
    return out

Q_BASE = """SELECT order_date_key, COUNT(*) AS orders, SUM(order_total_usd) AS revenue_usd, AVG(order_total_usd) AS avg_order_value
            FROM dw.fact_orders WHERE {salt} GROUP BY order_date_key"""
Q_MV   = "SELECT order_date_key, orders, revenue_usd, avg_order_value FROM dw.mv_daily_revenue WHERE {salt}"

t_base, t_mv = timed(Q_BASE), timed(Q_MV)
perf = pd.DataFrame({"query": ["aggregate dw.fact_orders", "read dw.mv_daily_revenue"],
                     "rows_scanned_source": [fact_counts["fact_orders"], mv_rows],
                     "first_run_ms (incl. compile)": [round(t_base[0], 1), round(t_mv[0], 1)],
                     "median_ms (runs 2-%d)" % RUNS: [round(float(np.median(t_base[1:])), 1), round(float(np.median(t_mv[1:])), 1)],
                     "min_ms": [round(min(t_base), 1), round(min(t_mv), 1)]})
display(perf)
speedup = float(np.median(t_base[1:])) / max(float(np.median(t_mv[1:])), 0.001)
print(f"Median speed-up of the MV over the base aggregation: {speedup:.2f}x "
      f"({fact_counts['fact_orders']:,} fact rows -> {mv_rows:,} MV rows)")
print("NOTE: with only ~2,500 fact rows both queries are very fast, so absolute timings are small and noisy. This measurement says nothing "
      "about larger data; what it does show is the reduction in rows scanned (see the plans below).")

def plan(sql):
    return "\n".join(str(list(r.values())[0]) for r in rs_exec("EXPLAIN " + sql, return_results=True))
print("\nEXPLAIN - base aggregation:\n", plan(Q_BASE.format(salt="order_date_key <> -99")))
print("\nEXPLAIN - materialized view:\n", plan(Q_MV.format(salt="order_date_key <> -99")))

# join colocation: DS_DIST_NONE in the plan means no data is redistributed for the join (colocated)
print("\nJoin colocation evidence (look for DS_DIST_NONE = no redistribution):")
join_checks = {
    "fact_orders JOIN dim_customer on customer_sk": "SELECT COUNT(*) FROM dw.fact_orders f JOIN dw.dim_customer c ON c.customer_sk = f.customer_sk",
    "fact_events JOIN dim_customer on customer_sk": "SELECT COUNT(*) FROM dw.fact_events f JOIN dw.dim_customer c ON c.customer_sk = f.customer_sk",
    "fact_graph_edges JOIN dim_product on to_product_sk": "SELECT COUNT(*) FROM dw.fact_graph_edges f JOIN dw.dim_product p ON p.product_sk = f.to_product_sk",
}
coloc_rows = []
for name, q in join_checks.items():
    p = plan(q)
    tokens = sorted(set(re.findall(r"DS_(?:DIST|BCAST)_\w+", p)))
    coloc_rows.append((name, ", ".join(tokens) or "(none)"))
COLOC = pd.DataFrame(coloc_rows, columns=["join", "distribution step in plan"])
display(COLOC)

# sort-key evidence: a date-range predicate on the sort key column
rng = rs_df("""SELECT MIN(order_date_key) AS lo, MAX(order_date_key) AS hi FROM dw.fact_orders""").iloc[0]
q_rng = f"SELECT COUNT(*), SUM(order_total_usd) FROM dw.fact_orders WHERE order_date_key BETWEEN {int(rng.lo)} AND {int(rng.lo) + 30}"
print("\nSort key evidence - EXPLAIN of a 31-day range filter on the SORTKEY column order_date_key:\n", plan(q_rng))

---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [ ]:
# ========= Data-quality checks (definitions; each query returns the number of OFFENDING rows, expected 0) =========
DQ_CHECKS = [
    # duplicates (Redshift does not enforce primary keys)
    ("duplicate order_id in stg.orders_raw",      "SELECT COUNT(*) FROM (SELECT order_id FROM stg.orders_raw GROUP BY 1 HAVING COUNT(*) > 1) d"),
    ("duplicate order_id in dw.fact_orders",      "SELECT COUNT(*) FROM (SELECT order_id FROM dw.fact_orders GROUP BY 1 HAVING COUNT(*) > 1) d"),
    ("duplicate event_id in dw.fact_events",      "SELECT COUNT(*) FROM (SELECT event_id FROM dw.fact_events GROUP BY 1 HAVING COUNT(*) > 1) d"),
    ("duplicate edge_id in dw.fact_graph_edges",  "SELECT COUNT(*) FROM (SELECT edge_id FROM dw.fact_graph_edges GROUP BY 1 HAVING COUNT(*) > 1) d"),
    ("duplicate customer_id in dim_customer",     "SELECT COUNT(*) FROM (SELECT customer_id FROM dw.dim_customer GROUP BY 1 HAVING COUNT(*) > 1) d"),
    ("duplicate product_id in dim_product",       "SELECT COUNT(*) FROM (SELECT product_id FROM dw.dim_product GROUP BY 1 HAVING COUNT(*) > 1) d"),
    # NULL business keys
    ("NULL order_id / customer_sk / order_date_key in fact_orders", "SELECT COUNT(*) FROM dw.fact_orders WHERE order_id IS NULL OR customer_sk IS NULL OR order_date_key IS NULL"),
    ("NULL event_id / customer_sk / event_date_key in fact_events", "SELECT COUNT(*) FROM dw.fact_events WHERE event_id IS NULL OR customer_sk IS NULL OR event_date_key IS NULL"),
    ("NULL edge_id / event_date_key in fact_graph_edges",           "SELECT COUNT(*) FROM dw.fact_graph_edges WHERE edge_id IS NULL OR event_date_key IS NULL"),
    # referential integrity (informational FKs are not enforced)
    ("fact_orders -> dim_date (order_date_key)",   "SELECT COUNT(*) FROM dw.fact_orders f LEFT JOIN dw.dim_date d ON d.date_key = f.order_date_key WHERE d.date_key IS NULL"),
    ("fact_orders -> dim_date (ship_date_key)",    "SELECT COUNT(*) FROM dw.fact_orders f LEFT JOIN dw.dim_date d ON d.date_key = f.ship_date_key WHERE f.ship_date_key IS NOT NULL AND d.date_key IS NULL"),
    ("fact_events -> dim_date",                    "SELECT COUNT(*) FROM dw.fact_events f LEFT JOIN dw.dim_date d ON d.date_key = f.event_date_key WHERE d.date_key IS NULL"),
    ("fact_graph_edges -> dim_date",               "SELECT COUNT(*) FROM dw.fact_graph_edges f LEFT JOIN dw.dim_date d ON d.date_key = f.event_date_key WHERE d.date_key IS NULL"),
    ("fact_orders -> dim_customer / channel / device / payment / shipping",
        "SELECT COUNT(*) FROM dw.fact_orders f WHERE f.customer_sk NOT IN (SELECT customer_sk FROM dw.dim_customer) OR f.channel_sk IS NULL OR f.device_sk IS NULL OR f.payment_method_sk IS NULL OR f.shipping_method_sk IS NULL"),
    ("fact_events with product in staging but NULL product_sk",
        "SELECT COUNT(*) FROM dw.fact_events f JOIN stg.events_raw s ON s.event_id = f.event_id WHERE s.product_id IS NOT NULL AND f.product_sk IS NULL"),
    ("fact_graph_edges: Customer from-node without from_customer_sk",
        "SELECT COUNT(*) FROM dw.fact_graph_edges f JOIN stg.edges_raw s ON s.edge_id = f.edge_id WHERE s.from_node_type = 'Customer' AND f.from_customer_sk IS NULL"),
    ("fact_graph_edges: Product to-node without to_product_sk",
        "SELECT COUNT(*) FROM dw.fact_graph_edges f JOIN stg.edges_raw s ON s.edge_id = f.edge_id WHERE s.to_node_type = 'Product' AND f.to_product_sk IS NULL"),
    ("fact_graph_edges: customer key filled on a PRODUCT node (routing error)",
        "SELECT COUNT(*) FROM dw.fact_graph_edges f JOIN stg.edges_raw s ON s.edge_id = f.edge_id WHERE (s.from_node_type = 'Product' AND f.from_customer_sk IS NOT NULL) OR (s.to_node_type = 'Product' AND f.to_customer_sk IS NOT NULL)"),
    # value quality
    ("negative order_total_usd / subtotal_usd",    "SELECT COUNT(*) FROM dw.fact_orders WHERE order_total_usd < 0 OR subtotal_usd < 0"),
    ("shipped before ordered",                     "SELECT COUNT(*) FROM dw.fact_orders WHERE ship_date_key < order_date_key"),
    ("discount_rate or tax_rate outside 0..1",     "SELECT COUNT(*) FROM dw.fact_orders WHERE discount_rate NOT BETWEEN 0 AND 1 OR tax_rate NOT BETWEEN 0 AND 1"),
    ("order total != subtotal-discount+shipping+tax (> 0.05)",
        "SELECT COUNT(*) FROM dw.fact_orders WHERE ABS(order_total_usd - (subtotal_usd - discount_amount_usd + shipping_cost_usd + tax_amount_usd)) > 0.05"),
    ("events: negative cart value / latency / dwell", "SELECT COUNT(*) FROM dw.fact_events WHERE cart_value_usd < 0 OR latency_ms < 0 OR dwell_seconds < 0"),
    ("edges: quantity <= 0 or unit_price <= 0",    "SELECT COUNT(*) FROM dw.fact_graph_edges WHERE quantity <= 0 OR unit_price_usd <= 0"),
    ("edges: edge_strength outside 0..1",          "SELECT COUNT(*) FROM dw.fact_graph_edges WHERE edge_strength NOT BETWEEN 0 AND 1"),
    ("unsupported relationship type",              "SELECT COUNT(*) FROM dw.fact_graph_edges WHERE relationship NOT IN ('VIEWED','PURCHASED','ADDED_TO_CART','ALSO_BOUGHT_WITH','REFERRED_FRIEND','RETURNS')"),
    # cross-source consistency
    ("edges.order_id not present in fact_orders",  "SELECT COUNT(*) FROM dw.fact_graph_edges f WHERE f.order_id IS NOT NULL AND f.order_id NOT IN (SELECT order_id FROM dw.fact_orders)"),
]


In [ ]:
print("="*60)
print("TASK 6: Validation and Reporting")
print("="*60)

print("\n📊 Row Counts:")
print("-" * 40)
tabs = [(r.schema, r.table) for _, r in DDL_TABLES.iterrows()]
union = " UNION ALL ".join(f"SELECT '{s}.{t}' AS table_name, COUNT(*) AS n FROM {s}.{t}" for s, t in tabs)
row_counts = rs_exec(f"SELECT * FROM ({union}) x ORDER BY 1", return_results=True)
for r in row_counts:
    print(f"  {r['table_name']:28s} {int(r['n']):>8,}")
ROW_COUNTS = pd.DataFrame(row_counts)

# ---- data-quality checks: each query returns the number of OFFENDING rows -> expected 0 ----------------------
print("\n🔎 Data quality checks (offending rows, expected 0):")
dq_rows = []
for name, sql in DQ_CHECKS:
    n = int(rs_scalar(sql))
    dq_rows.append((name, n, "PASS" if n == 0 else "FAIL"))
DQ = pd.DataFrame(dq_rows, columns=["check", "offending rows", "result"])
pd.set_option("display.max_colwidth", 100)
display(DQ)
print(f"{(DQ.result == 'PASS').sum()} of {len(DQ)} checks passed")
assert (DQ.result == "PASS").all(), "data quality failures - see table"

# informational cross-source facts (not pass/fail): how much do the three systems overlap?
overlap = rs_df("""
    SELECT (SELECT COUNT(DISTINCT customer_sk) FROM dw.fact_orders)  AS customers_with_orders,
           (SELECT COUNT(DISTINCT customer_sk) FROM dw.fact_events)  AS customers_with_events,
           (SELECT COUNT(DISTINCT o.customer_sk) FROM dw.fact_orders o JOIN dw.fact_events e ON e.customer_sk = o.customer_sk) AS customers_in_both,
           (SELECT COUNT(*) FROM dw.dim_customer) AS customers_total
""")
print("\nCross-source customer overlap (informational):")
display(overlap)

In [ ]:
print("\n📊 Sample Analytics - Daily Revenue:")
print("-" * 40)

# Q1 daily revenue, served from the materialized view
daily_rev = rs_df("""
    SELECT d.date_actual, mv.orders, mv.revenue_usd, mv.avg_order_value
    FROM dw.mv_daily_revenue mv
    JOIN dw.dim_date d ON d.date_key = mv.order_date_key
    ORDER BY mv.revenue_usd DESC
    LIMIT 10
""")
print("Q1  Top 10 days by revenue (from dw.mv_daily_revenue):")
display(daily_rev)

# Q1b: the same numbers computed from the fact table must agree with the view (no double counting)
chk_total = rs_df("SELECT SUM(mv.revenue_usd) AS mv_revenue, (SELECT SUM(order_total_usd) FROM dw.fact_orders) AS fact_revenue FROM dw.mv_daily_revenue mv")
display(chk_total)

# Q2 revenue by channel (fact_orders joined to dim_channel on channel_sk).
# The SQL text is kept in Q2_SQL so the very same string is executed here and printed in the final report (section 9).
Q2_SQL = """
SELECT
    ch.channel,
    COUNT(*)                                   AS orders,
    ROUND(SUM(f.order_total_usd), 2)           AS revenue_usd,
    ROUND(AVG(f.order_total_usd), 2)           AS avg_order_value,
    ROUND(100.0 * SUM(CASE WHEN f.returned THEN 1 ELSE 0 END) / COUNT(*), 1) AS return_rate_pct
FROM dw.fact_orders AS f
JOIN dw.dim_channel AS ch
    ON ch.channel_sk = f.channel_sk
GROUP BY ch.channel
ORDER BY revenue_usd DESC
""".strip()
q2 = rs_df(Q2_SQL)

# SQL (logical dw.* names) and a one-line explanation that the report prints next to the matching result table
ANALYTICS_SQL = {"Q2 Revenue by channel": Q2_SQL}
ANALYTICS_NOTE = {"Q2 Revenue by channel": "Which sales channels generate the most revenue, and how do their order values and return rates compare? "
                  "The query joins the order fact table to the channel dimension on `channel_sk` and aggregates per channel."}
print("\nQ2  Revenue by channel (shows which channels drive revenue and returns):")
display(q2)

# Q3 monthly revenue and discounting by payment method
q3 = rs_df("""
    SELECT pm.payment_method, COUNT(*) AS orders, ROUND(SUM(f.order_total_usd), 2) AS revenue_usd,
           ROUND(AVG(f.discount_rate) * 100, 2) AS avg_discount_pct
    FROM dw.fact_orders f JOIN dw.dim_payment_method pm ON pm.payment_method_sk = f.payment_method_sk
    GROUP BY pm.payment_method ORDER BY revenue_usd DESC
""")
print("\nQ3  Revenue and average discount by payment method:")
display(q3)

In [ ]:
# Q4 clickstream funnel by A/B variant (sessions reaching each step) - events only, no joins to orders
q4 = rs_df("""
    SELECT ab.ab_variant,
           COUNT(DISTINCT f.session_id)                                                       AS sessions,
           COUNT(DISTINCT CASE WHEN f.event_type = 'product_view'   THEN f.session_id END)    AS product_view,
           COUNT(DISTINCT CASE WHEN f.event_type = 'add_to_cart'    THEN f.session_id END)    AS add_to_cart,
           COUNT(DISTINCT CASE WHEN f.event_type = 'checkout_start' THEN f.session_id END)    AS checkout_start,
           COUNT(DISTINCT CASE WHEN f.event_type = 'purchase'       THEN f.session_id END)    AS purchase
    FROM dw.fact_events f JOIN dw.dim_ab_variant ab ON ab.ab_variant_sk = f.ab_variant_sk
    GROUP BY ab.ab_variant ORDER BY ab.ab_variant
""")
q4["purchase_per_1000_sessions"] = (1000 * q4.purchase / q4.sessions).round(1)
print("Q4  Funnel by A/B variant (sessions with at least one event of each type):")
display(q4)
print("    Reading: the variants are compared descriptively; with this sample size the difference is not evidence of a causal effect.")

# Q5 conversion-style view by referrer: how many events are purchases / add-to-carts
q5 = rs_df("""
    SELECT r.referrer, COUNT(*) AS events,
           SUM(CASE WHEN f.event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS add_to_cart_events,
           SUM(CASE WHEN f.event_type = 'purchase'    THEN 1 ELSE 0 END) AS purchase_events,
           ROUND(100.0 * SUM(CASE WHEN f.event_type = 'purchase' THEN 1 ELSE 0 END) / COUNT(*), 2) AS purchase_event_pct,
           ROUND(AVG(f.dwell_seconds), 1) AS avg_dwell_s
    FROM dw.fact_events f JOIN dw.dim_referrer r ON r.referrer_sk = f.referrer_sk
    GROUP BY r.referrer ORDER BY events DESC
""")
print("\nQ5  Engagement and purchase events by referrer:")
display(q5)

# Q6 product / category performance: orders (revenue) next to graph signals (views, purchases, returns) - each side aggregated
# separately BEFORE joining so revenue is never double counted
q6 = rs_df("""
    WITH rev AS (
        SELECT primary_category AS category, COUNT(*) AS orders, ROUND(SUM(order_total_usd), 2) AS revenue_usd
        FROM dw.fact_orders GROUP BY primary_category
    ), g AS (
        SELECT category,
               SUM(CASE WHEN relationship = 'VIEWED'    THEN 1 ELSE 0 END) AS views,
               SUM(CASE WHEN relationship = 'PURCHASED' THEN quantity ELSE 0 END) AS units_purchased,
               SUM(CASE WHEN relationship = 'RETURNS'   THEN 1 ELSE 0 END) AS return_edges
        FROM dw.fact_graph_edges GROUP BY category
    )
    SELECT rev.category, rev.orders, rev.revenue_usd, g.views, g.units_purchased, g.return_edges
    FROM rev LEFT JOIN g ON g.category = rev.category
    ORDER BY rev.revenue_usd DESC
""")
print("\nQ6  Category performance: orders/revenue (PostgreSQL) + graph behaviour (Neo4j):")
display(q6)

# Q7 most co-purchased products (graph) with their category and price
q7 = rs_df("""
    SELECT p.product_id, p.category, p.current_unit_price_usd, COUNT(*) AS also_bought_with_links
    FROM dw.fact_graph_edges f JOIN dw.dim_product p ON p.product_sk = f.from_product_sk
    WHERE f.relationship = 'ALSO_BOUGHT_WITH'
    GROUP BY p.product_id, p.category, p.current_unit_price_usd
    ORDER BY also_bought_with_links DESC, p.product_id LIMIT 10
""")
print("\nQ7  Products with the most ALSO_BOUGHT_WITH links (recommendation candidates):")
display(q7)

# Q8 customers present in both systems: orders next to clickstream activity (each side aggregated first)
q8 = rs_df("""
    WITH o AS (SELECT customer_sk, COUNT(*) AS orders, ROUND(SUM(order_total_usd), 2) AS revenue_usd FROM dw.fact_orders GROUP BY customer_sk),
         e AS (SELECT customer_sk, COUNT(*) AS events, SUM(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS add_to_cart_events
               FROM dw.fact_events GROUP BY customer_sk)
    SELECT c.customer_id, c.customer_segment, o.orders, o.revenue_usd, e.events, e.add_to_cart_events
    FROM o JOIN e ON e.customer_sk = o.customer_sk JOIN dw.dim_customer c ON c.customer_sk = o.customer_sk
    ORDER BY o.revenue_usd DESC LIMIT 10
""")
print("\nQ8  Customers appearing in BOTH orders and clickstream (top 10 by revenue):")
display(q8)
ANALYTICS = {"Q1 Top days by revenue (materialized view)": daily_rev, "Q2 Revenue by channel": q2, "Q3 Revenue by payment method": q3,
             "Q4 Funnel by A/B variant": q4, "Q5 Engagement by referrer": q5, "Q6 Category performance (orders + graph)": q6,
             "Q7 Most co-purchased products": q7, "Q8 Customers in orders and events": q8}

In [ ]:
print("\n📄 Generating Final Report...")

def df_to_md(df: pd.DataFrame) -> str:
    """Render a DataFrame as a GitHub-flavoured markdown table (no extra dependency)."""
    d = df.copy()
    d.columns = [str(c) for c in d.columns]
    head = "| " + " | ".join(d.columns) + " |\n|" + "|".join(["---"] * len(d.columns)) + "|\n"
    body = "".join("| " + " | ".join("" if (isinstance(v, float) and np.isnan(v)) else str(v).replace("|", "\\|") for v in row) + " |\n"
                   for row in d.itertuples(index=False, name=None))
    return head + body

REPORT_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%SZ")
log_df = pipeline_log_df()
log_md = df_to_md(log_df[["stage", "rows_in", "rows_out", "status", "seconds"]].fillna("")) if len(log_df) else "_no stages logged_"

FENCE = "`" * 3          # markdown code fence (built here so the cell contains no literal triple backticks)
with open(MERMAID_MD) as f:
    mermaid_block = re.search(FENCE + r"mermaid(.*?)" + FENCE, f.read(), flags=re.S).group(1).strip()

sources_md = df_to_md(pd.DataFrame([("PostgreSQL", "orders (raw.orders)", len(orders_df), orders_df.shape[1]),
                                     ("Cassandra", "clickstream events (events)", len(events_df), events_df.shape[1]),
                                     ("Neo4j", "graph edges (relationships)", len(edges_df), edges_df.shape[1])],
                                    columns=["System", "Content", "Records", "Columns"]))
schema_md = df_to_md(DDL_TABLES.drop(columns=["columns", "has_identity"]))
fact_rows_total = int(ROW_COUNTS[ROW_COUNTS.table_name.str.contains("fact_")].n.astype(int).sum())
dq_pass, dq_total = int((DQ.result == "PASS").sum()), len(DQ)
mapping_md, recon_md, tot_md, dq_md = df_to_md(FIELD_MAP), df_to_md(recon), df_to_md(tot), df_to_md(DQ)
overlap_md, design_md, perf_md, coloc_md = df_to_md(overlap), df_to_md(DESIGN_TABLE), df_to_md(perf), df_to_md(COLOC)
def analytics_section(k, df):
    """### heading, optional question + fenced SQL (physical names for the active naming mode), then the real result table."""
    parts = ["### " + k + "\n"]
    if k in ANALYTICS_SQL:
        parts.append("**What it answers:** " + ANALYTICS_NOTE[k] + "\n")
        parts.append(FENCE + "sql\n" + _apply_naming(ANALYTICS_SQL[k]) + "\n" + FENCE + "\n")
        parts.append("**Result:**\n")
    parts.append(df_to_md(df) + "\n")
    return "\n".join(parts)

analytics_md = "\n".join(analytics_section(k, v) for k, v in ANALYTICS.items())

naming_note = ("**Naming note:** this database user is not allowed to create schemas, so every logical name `stg.<table>` / `dw.<table>` used in this report "
               "is physically stored as `public.stg_<table>` / `public.dw_<table>` (for example `dw.fact_orders` is `public.dw_fact_orders`). "
               "The mapping was applied to every statement, never mixed." if NAMING["public_fallback"]
               else "**Naming note:** objects live in the `stg` and `dw` schemas exactly as named in the DDL.")

report_content = f"""# Data Warehouse Build Report

Generated: {now} (all numbers below were produced by this run of the notebook against Amazon Redshift)

## 1. Executive Summary
An e-commerce company keeps orders in **PostgreSQL**, clickstream events in **Apache Cassandra** and customer/product
relationships in **Neo4j**, so revenue, behaviour and product questions cannot be answered consistently. This project loads the three
sources through their native Python drivers, transforms and conforms the data, stages it in **Amazon Redshift** (`stg` schema) and
builds a star schema (`dw` schema) with 12 dimensions and 3 fact tables, optimized with distribution keys, sort keys,
compression and a materialized view. Result: {fact_rows_total:,} fact rows loaded,
no records lost between source and fact, {dq_pass}/{dq_total} data-quality checks passed.

## 2. Source Systems
{sources_md}

## 3. Source-to-Target Mapping
{mapping_md}

**ID standardization:** customer_id (`C#####`) and product_id (`P####`) share one format across the systems, so they are trimmed,
upper-cased, pattern-validated and used as the natural keys of `dim_customer` / `dim_product`. Graph node ids are routed by
`*_node_type` (Customer -> dim_customer, Product -> dim_product), never assumed to be customers. `order_id` links orders and graph edges.
Missing optional keys stay NULL; missing critical keys abort the load.

## 4. Dimensional Schema
{naming_note}

{schema_md}

**Grain:** `fact_orders` = one row per order; `fact_events` = one row per clickstream event; `fact_graph_edges` = one row per graph edge.
**Surrogate keys:** `IDENTITY` columns (`customer_sk`, `product_sk`, `channel_sk`, ...); `dim_date.date_key` = YYYYMMDD.
**SCD:** customer/product dimensions carry `effective_from/effective_to/is_current` and are loaded as one current row per key
(Type-2 ready, not full history tracking).

## 5. Schema Diagram
![Schema diagram](schema-diagram.svg)

<details><summary>Mermaid source (matches the DDL, verified programmatically)</summary>

{FENCE}mermaid
{mermaid_block}
{FENCE}
</details>

## 6. ETL/ELT Implementation
1. **Source loading** - CSV -> PostgreSQL (`psycopg2.extras.execute_values`, `ON CONFLICT DO NOTHING`), Cassandra (prepared statements,
   `execute_concurrent_with_args`), Neo4j (`UNWIND` + `MERGE`, whitelisted labels/relationship types, node uniqueness constraints).
2. **Extraction** - SQLAlchemy/pandas, cassandra-driver, neo4j driver; values compared cell-by-cell with the CSV.
3. **Transformation** - schema-drift check, typed conversion (timestamps, ints, floats, booleans), trimming/case rules, rounding to DDL precision,
   identifier validation, duplicate/NULL-key rejection; DataFrames match the staging column specifications.
4. **Staging** - Redshift Data API (`rs_exec`, `rs_batch_insert`); staging tables are truncated and reloaded so re-runs are safe.
5. **Dimensions/facts** - set-based SQL; every insert is guarded by NOT EXISTS so re-running adds nothing twice.
6. **Error handling/logging** - failures raise (no false success), expired AWS credentials are distinguished from SQL errors, and every stage is logged:

{log_md}

## 7. Data Quality
{recon_md}

{tot_md}

{dq_md}

Cross-source customer overlap:

{overlap_md}
Only a small share of customers appears in more than one operational system, so cross-system customer analysis covers just those customers.

## 8. Redshift Optimization
{design_md}

**Compression:** `ENCODE zstd` on most columns, `ENCODE raw` on leading sort-key columns.
**Materialized view:** `dw.mv_daily_revenue` ({mv_rows:,} rows) - orders, revenue and average order value per day; identical to a fresh aggregation of `fact_orders` (0 differing rows) and refreshed with `REFRESH MATERIALIZED VIEW`.
**ANALYZE** was run on the fact tables, `dim_customer`, `dim_product` and `dim_date`.

**Measured query time (Redshift-reported, {RUNS} runs, result cache bypassed):**

{perf_md}

Median speed-up of the view over the base aggregation: **{speedup:.2f}x**. With roughly 2,500 fact rows absolute times are tiny and noisy, so this
shows direction rather than production-scale gains.

**Join colocation (from EXPLAIN):**

{coloc_md}

## 9. Analytical Queries (real results from Redshift)
{analytics_md}
## 10. Limitations and Improvements
- Small dataset (2,500 rows per source): performance differences are small; sort/dist benefits would show at larger scale.
- Batch `INSERT` loading is used because the project supplies it; production loads should use `COPY` from S3.
- Loads are re-runnable: staging is truncated and reloaded; new keys are inserted and changed values of existing keys are overwritten in place (Type 1). Source deletions are not propagated (the load stops with an error) and there is no CDC or late-arriving-data logic.
- Dimensions are SCD-Type-2 *ready* only (effective_from / effective_to / is_current exist); historical version tracking is not implemented: one current row per customer/product, changed attributes are overwritten.
- Customer overlap between the three systems is small, which limits cross-system customer analytics.
- Redshift PRIMARY/FOREIGN KEYs are informational; integrity is enforced by the validation queries above.
- Lab AWS credentials expire after about 4 hours; the notebook reports expiry separately from SQL errors.
"""

report_path = os.path.join(REPORT_DIR, "warehouse_report.md")
with open(report_path, "w") as f:
    f.write(report_content)
with open(os.path.join(REPORT_DIR, "field-mapping.md"), "w") as f:
    f.write("# Source-to-Target Field Mapping\n\n" + df_to_md(FIELD_MAP))
with open(os.path.join(REPORT_DIR, "validation-results.md"), "w") as f:
    f.write("# Validation Results\n\n## Reconciliation\n" + df_to_md(recon) + "\n## Control totals\n" + df_to_md(tot) +
            "\n## Data-quality checks\n" + df_to_md(DQ) + "\n## Row counts\n" + df_to_md(ROW_COUNTS) + "\n")
log_df.to_csv(os.path.join(REPORT_DIR, "pipeline_log.csv"), index=False)

print(f"\n✅ Report saved to: {report_path}")
print("   also: reports/field-mapping.md, reports/validation-results.md, reports/pipeline_log.csv")
display(log_df[["stage", "rows_in", "rows_out", "status", "seconds"]])
failed = log_df[log_df.status != "OK"]
print("\n" + "="*60)
print("PIPELINE FINISHED - all logged stages OK" if failed.empty else f"PIPELINE FINISHED WITH FAILED STAGES: {failed.stage.tolist()}")
print("="*60)